### The following code is related to Fig.2c, Fig.2g, Fig.2j, Fig.3h, Fig.3j, and Supplementary Fig.10d.

In [ ]:
library(tibble)
library(stringr)
library(reshape2)
library(tidyr)
library(furrr)
library(future)
library(dplyr)
library(Seurat)
library(ggplot2)
library(clusterProfiler)
library(org.Hs.eg.db)

In [ ]:
setwd('/mnt/data/khm_scRNA/huh7_zxs/')
seurat_obj <- readRDS('./result_zxs/scdata_filter.rds')

## data reading

In [ ]:
seurat_obj <- subset(seurat_obj,subset = batch %in% c('normal2','Tatin'))
seurat_obj

In [ ]:
meta_data <- seurat_obj@meta.data %>% 
    mutate(
        batch = ifelse(batch == 'Tatin',yes = 'Tatin',no = 'Normal'),
        function_type = case_when(
            sgRNA_type %in% c("SREBF2", "HMGCR", "SQLE", "INSIG1") ~ 'chol_synthesis',
            sgRNA_type %in% c("LDLR", "NPC1L1", "NPC1") ~ 'chol_uptake',
            sgRNA_type %in% c("APOB", "MTTP", "ABCA1", "ABCG1", "ABCG5", "ABCG8", "NR1H3") ~ 'chol_efflux',
            sgRNA_type %in% c("SOAT1") ~ 'chol_esterification',
            sgRNA_type %in% c("AAVS",'NT') ~ 'NT',
            TRUE ~ sgRNA_type
        ),
        sgRNA_type_use = ifelse(sgRNA_type %in% c("AAVS",'NT'),yes = 'NT',no = sgRNA_type),
        sgRNA_type_use = factor(sgRNA_type_use,levels = c(
            "NT","SREBF2","HMGCR","SQLE","INSIG1","LDLR","NPC1L1","NPC1","APOB",
            "MTTP","ABCA1","ABCG1","ABCG5","ABCG8","NR1H3","SOAT1"
        ))
    )
seurat_obj@meta.data <- meta_data
meta_data$batch %>% table()
meta_data$sgRNA_type_use %>% unique()

In [ ]:
seurat_obj$batch %>% table()

In [ ]:
seurat_obj

## all peotein expression within batch

In [ ]:
DefaultAssay(seurat_obj) <- 'Protein'
seurat_obj <- NormalizeData(object = seurat_obj,normalization.method = "CLR", margin =2)
seurat_obj

In [ ]:
feature_use <- rownames(seurat_obj)
feature_use

In [ ]:
seurat_obj@meta.data %>% colnames()

In [ ]:
options(repr.plot.width = 32,repr.plot.height = 12)
data_plot <- FetchData(
    object = seurat_obj,
    vars = c(feature_use,'batch','sgRNA_type'),
    layer = "data"
) %>% 
    rownames_to_column('sample') %>% 
    pivot_longer(cols = all_of(feature_use))
data_plot %>% head()

In [ ]:
data_plot$sgRNA_type %>% unique() %>% length()

In [ ]:
data_plot$batch %>% unique()

In [ ]:
library(ggpubr)
options(repr.plot.width = 32,repr.plot.height = 7)
p_allProtein <- lapply(data_plot$name %>% unique(),FUN = function(sgRNA_select){
    p <- ggboxplot(
      data_plot %>% filter(name == sgRNA_select),
      x="batch", y="value",
      color ="batch",
      width = 0.6,
      # palette = paletteer_d("ggsci::nrc_npg"),
      add = "jitter",
      xlab = F,  bxp.errorbar=T,
      bxp.errorbar.width=0.5, 
      size=0, outlier.shape=NA,legend = "right") +
      guides(color = guide_legend(title = 'Group'))+
      stat_compare_means(
        label = "p.format",size = 8,
        comparisons = list(c(
          "normal2","Tatin"
        )),
        method = "wilcox.test",
      ) +
      facet_wrap(~sgRNA_type,ncol = 17,strip.position = 'left',scales = 'free') +
      labs(title = sgRNA_select) +
      theme(
        plot.title = element_text(hjust = 0.5,size = 32),
        legend.title = element_text(size = 24),
        legend.text = element_text(size = 20),
        legend.key.height = unit(1.2, "cm"),
        legend.key.width = unit(1.2, "cm"),
        axis.text.x = element_blank(),
        axis.title.y = element_blank(),
        strip.background = element_blank(),
        strip.text = element_text(size = 20,face = 'italic',vjust = 0.7),
        strip.placement = 'outside'
      )
    return(p)
})

In [ ]:
p_allProtein

### Analysis of sgRNA identity composition in the top/bottom 20% or 30% of cells ranked by ALOD4 protein expression

In [ ]:
DefaultAssay(seurat_obj) <- 'Protein'
seurat_obj <- NormalizeData(object = seurat_obj,normalization.method = "CLR", margin =2)
seurat_obj

In [ ]:
feature <- rownames(seurat_obj)
feature

In [ ]:
feature_use <- 'ALOD4-pAbO'

In [ ]:
seurat_obj@meta.data %>% colnames()

In [ ]:
data_exp <- FetchData(
    object = seurat_obj,
    vars = c(feature_use,'batch','sgRNA_type'),
    layer = "data"
) %>% 
    rownames_to_column('sample') %>% 
    pivot_longer(cols = all_of(feature_use)) %>% 
    dplyr::select(c('sample','value'))
data_exp %>% head()

In [ ]:
meta_data <- seurat_obj@meta.data  %>% 
    rownames_to_column('sample') %>% 
    left_join(data_exp,by = 'sample') %>% 
    column_to_rownames('sample') %>% 
    arrange(desc(value)) %>% 
    mutate(
        ratio_alod4 = 1:n(),
        group_alod4_30 = case_when(
            ratio_alod4 <= (n()*0.3) ~ 'top 30%',
            ratio_alod4 > (n()*0.7) ~ 'bottom 30%',
            TRUE ~ 'middle'
        ),
        group_alod4_20 = case_when(
            ratio_alod4 <= (n()*0.2) ~ 'top 20%',
            ratio_alod4 > (n()*0.8) ~ 'bottom 80%',
            TRUE ~ 'middle'
        )
    )
meta_data$group_alod4_20 %>% table()
meta_data$group_alod4_30 %>% table()
meta_data %>% head()

In [ ]:
ggplot(meta_data, aes(x = value,group = 1)) +
  geom_density(aes(fill = group_alod4_20)) +
  labs(title = "Diamond price density",
       x = "Price (USD)",
       y = "Density")

In [ ]:
dens <- density(meta_data$value, bw = "nrd0")
edges <- dens$x
dens_df <- data.frame(x = dens$x, y = dens$y)

df_long <- meta_data %>%
  mutate(
    bin = cut(value, breaks = edges, include.lowest = TRUE),
    w = 1
  )

prop_df <- df_long %>%
  group_by(bin, group_alod4_30) %>%
  summarise(w = sum(w), .groups = "drop") %>%
  group_by(bin) %>%
  mutate(prop = w / sum(w)) %>%
  ungroup() %>%
  mutate(bin = as.character(bin)) 

plot_df <- dens_df %>%
  mutate(bin = cut(x, breaks = edges, include.lowest = TRUE),
         bin = as.character(bin)) %>%
  left_join(prop_df, by = "bin") %>%
  mutate(height = y * prop) %>% 
  filter(!is.na(prop), !is.na(group_alod4_30))
vline_df <- meta_data %>%
    group_by(group_alod4_30) %>%
    summarise(xintercept = max(value)) %>% 
    mutate(color = paletteer::paletteer_d("ggsci::default_jama")[1:n()]) %>% 
    filter(group_alod4_30 != 'top 30%')
options(repr.plot.width = 16,repr.plot.height = 7)
ggplot(plot_df, aes(x = x, y = height, fill = group_alod4_30)) +
    geom_area(position = "stack", colour = NA,alpha = 0.4) +
    geom_density(data = meta_data, aes(x = value, y = after_stat(density)), 
               color = "black", size = 1, inherit.aes = FALSE) +
    geom_segment(data = vline_df,
               aes(x = xintercept, xend = xintercept, y = 0, yend = 0.55),  
               color = vline_df$color,
               linetype = "dashed", size = 1.2, inherit.aes = FALSE) +
    scale_x_continuous(expand = c(0,0)) +
    scale_y_continuous(expand = c(0,0)) +
    annotate(geom = 'text',x = 0.9,y = 0.5,label = 'Bottom 30%',size = 6) +
    annotate(geom = 'text',x = 2.7,y = 0.5,label = 'Top 30%',size = 6) +
    scale_fill_manual(values = paletteer::paletteer_d("ggsci::default_jama")) +
    labs(x = "CLR-transformed ADT counts", y = "Density",fill = "Group") +
    theme_minimal() +
    theme(
        panel.grid = element_blank(),
        axis.line = element_line(color = 'black'),
        legend.position = 'none',
        axis.text = element_text(size = 20),
        axis.title = element_text(size = 24)
    )

In [ ]:
data_cell <- meta_data %>% 
    mutate(
        sgRNA_type = case_when(
            sgRNA_type %in% c('NT','AAVS') ~ 'NT',
            TRUE ~ sgRNA_type
        )
    )
data_cell %>% head()

In [ ]:
chisq <- chisq.test(table(data_cell$group_alod4_30,data_cell$sgRNA_identity))
data_plot <- chisq$observed/chisq$expected %>% 
    as.matrix() %>% as.data.frame()
data_plot %>% head()

In [ ]:
std_resid <- chisq$residuals
p_matrix <- 2 * (1 - pnorm(abs(std_resid)))

p_df <- as.data.frame(p_matrix) %>%
  rename_all(~c('celltype','sgRNA_type','p_value'))

data_ggplot <- data_plot %>%
  tibble::rownames_to_column('celltype') %>%
  reshape2::melt(id.var = 'celltype',variable.name = 'sgRNA_type',value.name = 'Ro/e') %>% 
  group_by(celltype) %>% 
  mutate(
      roe_scale = `Ro/e`
  )
data_ggplot <- data_ggplot %>%
  left_join(p_df, by = c("celltype", "sgRNA_type")) %>%
  mutate(
      sig = ifelse(p_value < 0.05, "(*)", ""),
      label = paste(round(`Ro/e`,digits = 2),sig,sep = '')
  )
data_ggplot %>% head()

In [ ]:
# data_ggplot$roe_scale <- data_ggplot$`Ro/e`
limit_min <- data_ggplot$roe_scale %>% min()
limit_max <- data_ggplot$roe_scale %>% max()
options(repr.plot.width = 6,repr.plot.height = 18)
ggplot(data = data_ggplot,
       mapping = aes(x = sgRNA_type,y = celltype)) +
  geom_tile(aes(fill = roe_scale)) +
  geom_text(aes(label = label)) +
  guides(
    fill = guide_colorbar(title = 'Ro/e',title.vjust = 1)
  ) +
  scale_fill_gradient(low = 'grey90',high =  '#3131F2',limit = c(limit_min,limit_max)) +
  theme_bw() +
  coord_flip() +
  theme(
    panel.grid = element_blank(),
    panel.border = element_blank(),
    axis.title = element_blank(),
    axis.line = element_blank(),
    axis.ticks = element_blank(),
    axis.text = element_text(size = 20),
    axis.text.x = element_text(angle = 30,hjust = 1),
    axis.text.y = element_text(hjust = 1),
    legend.text = element_text(size = 16),
    legend.position = 'right',
    legend.justification = c(0,1),
    legend.title = element_text(size = 20),
    strip.text = element_blank(),
    strip.background = element_blank()
  )

#### Calculate differential genes between the top and bottom groups

In [ ]:
seurat_obj
meta_data %>% dim()
meta_data %>% head()

In [ ]:
seurat_obj@meta.data <- meta_data
seurat_obj_subset <- subset(seurat_obj,subset = group_alod4_30 %in% c('top 30%','bottom 30%'))
seurat_obj_subset
DefaultAssay(seurat_obj_subset) <- 'RNA'

In [ ]:
diffgene <- FindMarkers(
    object = seurat_obj_subset,
    slot = 'data',
    group.by = 'group_alod4_30',
    ident.1	= 'top 30%',
    min.pct = 0.01,
    random.seed	= 1234,
    logfc.threshold = 0.1,
    only.pos = FALSE
) %>% 
    mutate(
        group = case_when(
            (p_val<0.05) & (avg_log2FC > 0.5) ~ 'Up in Top 30%',
            (p_val<0.05) & (avg_log2FC < -0.5) ~ 'Down in Top 30%',
            TRUE ~ 'Stable'
        )
    )
diffgene$group %>% table()
diffgene %>% head()

In [ ]:
cut_off_logFC <- 0.5
cut_off_pvalue <- 0.05
options(repr.plot.width = 12,repr.plot.height = 7)
ggplot(diffgene, aes(x = avg_log2FC, y = -log10(p_val), colour=group)) +
    geom_point(alpha=0.4, size=3.5) +
    scale_color_manual(values=c("#546de5", "#d2dae2","#ff4757"))+
    geom_vline(
        xintercept=c(-cut_off_logFC,cut_off_logFC),
        lty=4, col="black", lwd=0.8) +
    geom_hline(yintercept = -log10(cut_off_pvalue),
        lty=4, col="black", lwd=0.8) +
    labs(x="log2(fold change)", y="-log10 (p-value)")+
    theme_bw()+
    theme(
        legend.position="right",
        legend.title = element_blank()
    )

In [ ]:
library(org.Hs.eg.db)
library(clusterProfiler)

In [ ]:
diffgene$group %>% unique()

In [ ]:
de_res <- list(
    'top30' = diffgene %>% filter(group == 'Up in Top 30%') %>% rownames(),
    'bottom30' = diffgene %>% filter(group == 'Down in Top 30%') %>% rownames()
)
names(de_res)

In [ ]:
go_res <- lapply(names(de_res),FUN = function(sgRNA_identity){
    enrich.go <- enrichGO(
      gene = de_res[[sgRNA_identity]],
      OrgDb = 'org.Hs.eg.db',
      keyType = 'SYMBOL',
      ont = 'BP',
      pAdjustMethod = 'fdr',
      pvalueCutoff = 1,
      qvalueCutoff = 1,
      readable = FALSE
    )
    result <- enrich.go@result %>% 
        tidyr::separate(col = GeneRatio,into = c('targetgene','allgene'),sep = '/') %>% 
        mutate(
            targetgene = targetgene %>% as.numeric(),
            allgene = allgene %>% as.numeric(),
            generatio = targetgene/allgene,
            category = case_when(
                p.adjust<0.05 ~ 'Sig(p.adjust)',
                pvalue<0.05 ~ 'Sig(pvalue)',
                TRUE ~ 'N.S.'
            )
        ) %>% 
        tidyr::unite(col = GeneRatio,targetgene,allgene,sep = '/') %>% 
        arrange(desc(generatio))
    return(list(go_result = enrich.go,df_res = result))
})

In [ ]:
names(go_res) <- names(de_res)

In [ ]:
lapply(names(go_res),function(x){
    print(x)
    go_res[[x]]$df_res %>% dim() %>% print()
})

In [ ]:
getwd()

In [ ]:
saveRDS(go_res,'./result_zxs/go_res_alod4.rds')

### Fig3 h

In [ ]:
DefaultAssay(seurat_obj) <- 'Protein'
seurat_obj <- NormalizeData(object = seurat_obj,normalization.method = "CLR", margin =2)
seurat_obj

In [ ]:
all(colnames(seurat_obj) == colnames(seurat_obj[["Protein"]]))
all(colnames(seurat_obj) ==  colnames(seurat_obj[["RNA"]]))

In [ ]:
gene_vec <- c(
    "SREBF2-sg1","SREBF2-sg3","INSIG1-sg1","LDLR-sg1","LDLR-sg2","LDLR-sg3","NOC1L1-sg2",
    "NPC1-sg1","NPC1-sg2","NPC1-sg3","APOB-sg2","APOB-sg3","ABCA1-sg1","ABCA1-sg2","ABCG1-sg1",
    "ABCG1-sg2","ABCG1-sg3","ABCG5-sg2","ABCG5-sg3","ABCG8-sg2","SOAT1-sg1","SOAT1-sg2",
    "SOAT1-sg3","HMGCR-sg1","HMGCR-sg3","SQLE-sg3","MTTP-sg2",
    'NT1','NT2','AAVS'
)
seurat_use <- subset(seurat_obj,subset = sgRNA_identity %in% gene_vec)
seurat_use <- subset(seurat_use,subset = batch %in% c('Tatin'))

In [ ]:
all(colnames(seurat_use) == colnames(seurat_use[["Protein"]]))
all(colnames(seurat_use) ==  colnames(seurat_use[["RNA"]]))

In [ ]:
seurat_use$batch %>% unique()

In [ ]:
feature <- rownames(seurat_use)
feature

In [ ]:
feature_use <- 'ALOD4-pAbO'

In [ ]:
seurat_use@meta.data %>% colnames()

In [ ]:
data_exp <- FetchData(
    object = seurat_use,
    vars = c(feature_use,'batch','sgRNA_type'),
    layer = "data"
) %>% 
    rownames_to_column('sampleid') %>% 
    pivot_longer(cols = all_of(feature_use)) %>% 
    dplyr::select(c('sampleid','value'))
data_exp %>% head()

In [ ]:
all(colnames(seurat_use) == colnames(seurat_use[["Protein"]]))
all(colnames(seurat_use) ==  colnames(seurat_use[["RNA"]]))

In [ ]:
sampleid_lv <- colnames(seurat_use)
meta_data <- seurat_use@meta.data  %>% 
    rownames_to_column('sampleid') %>% 
    left_join(data_exp,by = 'sampleid') %>% 
    mutate(sampleid = factor(sampleid,levels = sampleid_lv)) %>% 
    arrange(desc(value)) %>% 
    mutate(
        ratio_alod4 = 1:n(),
        group_alod4_30 = case_when(
            ratio_alod4 <= (n()*0.2) ~ 'top 20%',
            ratio_alod4 > (n()*0.8) ~ 'bottom 20%',
            TRUE ~ 'middle'
        ),
        group_alod4_20 = case_when(
            ratio_alod4 <= (n()*0.2) ~ 'top 20%',
            ratio_alod4 > (n()*0.8) ~ 'bottom 20%',
            TRUE ~ 'middle'
        )
    ) %>% 
    arrange(sampleid) %>% 
    mutate(sampleid = sampleid %>% as.character()) %>% 
    column_to_rownames('sampleid')
table(rownames(meta_data) == colnames(seurat_use))
seurat_use@meta.data <- meta_data
meta_data$group_alod4_20 %>% table()
meta_data$group_alod4_30 %>% table()
meta_data %>% head()

In [ ]:
all(colnames(seurat_use) == colnames(seurat_use[["RNA"]]))

In [ ]:
options(repr.plot.width = 8,repr.plot.height = 4)
ggplot(meta_data, aes(x = value,group = 1)) +
  geom_density(aes(fill = group_alod4_20)) +
  labs(title = "Diamond price density",
       x = "Price (USD)",
       y = "Density")


In [ ]:
meta_data$batch %>% table()
meta_data %>% head()

In [ ]:
meta_data <- meta_data %>% arrange(desc(value))
dens <- density(meta_data$value, bw = "nrd0")
edges <- dens$x
dens_df <- data.frame(x = dens$x, y = dens$y)

df_long <- meta_data %>%
  mutate(
    bin = cut(value, breaks = edges, include.lowest = TRUE),
    w = 1
  )

prop_df <- df_long %>%
  group_by(bin, group_alod4_30) %>%
  summarise(w = sum(w), .groups = "drop") %>%
  group_by(bin) %>%
  mutate(prop = w / sum(w)) %>%
  ungroup() %>%
  mutate(bin = as.character(bin))  

plot_df <- dens_df %>%
  mutate(bin = cut(x, breaks = edges, include.lowest = TRUE),
         bin = as.character(bin)) %>% 
  left_join(prop_df, by = "bin") %>%
  mutate(height = y * prop) %>% 
  filter(!is.na(prop), !is.na(group_alod4_30))
vline_df <- meta_data %>%
    group_by(group_alod4_30) %>%
    summarise(xintercept = max(value)) %>% 
    mutate(color = c("#3f72af","#dbe2ef","#3f72af")) %>% 
    filter(group_alod4_30 != 'top 20%')
options(repr.plot.width = 16,repr.plot.height = 7)
ggplot(plot_df, aes(x = x, y = height, fill = group_alod4_30)) +
    geom_area(position = "stack", colour = NA,alpha = 0.4) +
    geom_density(data = meta_data, aes(x = value, y = after_stat(density)), 
               color = "black", size = 1, inherit.aes = FALSE) +
    geom_segment(data = vline_df,
               aes(x = xintercept, xend = xintercept, y = 0, yend = 0.55), 
               color = "#3f72af",
               linetype = "dashed", size = 1.2, inherit.aes = FALSE) +
    scale_x_continuous(expand = c(0,0)) +
    scale_y_continuous(expand = c(0,0)) +
    annotate(geom = 'text',x = 0.5,y = 0.5,label = 'Bottom 20%',size = 6) +
    annotate(geom = 'text',x = 2.7,y = 0.5,label = 'Top 20%',size = 6) +
    scale_fill_manual(values = c("#3f72af","#dbe2ef","#3f72af")) +
    labs(x = "CLR-transformed ADT counts", y = "Density",fill = "Group") +
    theme_minimal() +
    theme(
        panel.grid = element_blank(),
        axis.line = element_line(color = 'black'),
        legend.position = 'none',
        axis.ticks = element_line(),
        axis.text = element_text(size = 20),
        axis.title = element_text(size = 24)
    )
ggsave('./result_figs/Density_Alod4_group.pdf',width = 16,height = 7)

In [ ]:
data_cell <- meta_data %>% 
    mutate(
        sgRNA_type = case_when(
            sgRNA_type %in% c('NT','AAVS') ~ 'NT',
            TRUE ~ sgRNA_type
        ),
        sgRNA_identity = case_when(
            sgRNA_identity %in% c('NT1','NT2','AAVS') ~ 'NT',
            TRUE ~ sgRNA_identity
        )
    )
data_cell %>% head()

In [ ]:
data_cell %>% head()

In [ ]:
chisq <- chisq.test(table(data_cell$group_alod4_30,data_cell$batch))
data_plot <- chisq$observed/chisq$expected %>% 
    as.matrix() %>% as.data.frame()
data_plot %>% head()

In [ ]:
chisq <- chisq.test(table(data_cell$group_alod4_30,data_cell$sgRNA_identity))
data_plot <- chisq$observed/chisq$expected %>% 
    as.matrix() %>% as.data.frame()
data_plot %>% head()

In [ ]:
std_resid <- chisq$residuals
p_matrix <- 2 * (1 - pnorm(abs(std_resid)))

p_df <- as.data.frame(p_matrix) %>%
  rename_all(~c('celltype','sgRNA_type','p_value'))

data_ggplot <- data_plot %>%
  tibble::rownames_to_column('celltype') %>%
  reshape2::melt(id.var = 'celltype',variable.name = 'sgRNA_type',value.name = 'Ro/e') %>% 
  group_by(celltype) %>% 
  mutate(
      roe_scale = `Ro/e`
  )
data_ggplot <- data_ggplot %>%
  left_join(p_df, by = c("celltype", "sgRNA_type")) %>% 
    group_by(celltype) %>% 
    mutate(
        roe_NT = `Ro/e`[sgRNA_type == "NT"], 
        roe_scale = `Ro/e` / roe_NT
    ) %>% dplyr::select(-roe_NT) %>%
  mutate(
      sig = ifelse(p_value < 0.05, "(*)", ""),
      label = paste(round(roe_scale,digits = 2),sig,sep = '')
  )
data_ggplot %>% head()

In [ ]:
limit_min <- data_ggplot$roe_scale %>% min()
limit_max <- data_ggplot$roe_scale %>% max()
options(repr.plot.width = 6,repr.plot.height = 18)
ggplot(data = data_ggplot,
       mapping = aes(x = sgRNA_type,y = celltype)) +
  geom_tile(aes(fill = roe_scale)) +
  geom_text(aes(label = label)) +
  guides(
    fill = guide_colorbar(title = 'Ro/e',title.vjust = 1)
  ) +
  scale_fill_gradient(low = 'grey90',high =  '#3131F2',limit = c(limit_min,limit_max)) +
  theme_bw() +
  coord_flip() +
  theme(
    panel.grid = element_blank(),
    panel.border = element_blank(),
    axis.title = element_blank(),
    axis.line = element_blank(),
    axis.ticks = element_blank(),
    axis.text = element_text(size = 20),
    axis.text.x = element_text(angle = 30,hjust = 1),
    axis.text.y = element_text(hjust = 1),
    legend.text = element_text(size = 16),
    legend.position = 'right',
    legend.justification = c(0,1),
    legend.title = element_text(size = 20),
    strip.text = element_blank(),
    strip.background = element_blank()
  )

In [ ]:
colorRampPalette(colors = c('grey90','#26456E'))(5)


In [ ]:
targets <- c(
    "NPC1-sg2","NPC1-sg3","SREBF2-sg3",
    "SOAT1-sg1","SOAT1-sg3","SOAT1-sg2",
    "LDLR-sg1","LDLR-sg2","LDLR-sg3"
)
data_ggplot_sub <- data_ggplot %>% 
    filter(sgRNA_type %in% targets) %>% 
    mutate(sgRNA_type = factor(sgRNA_type,levels = targets))
limit_min <- data_ggplot_sub$roe_scale %>% min()
limit_max <- data_ggplot_sub$roe_scale %>% max()
limit_mean <- (limit_max+limit_min)/2
options(repr.plot.width = 6,repr.plot.height = 8)
ggplot(data = data_ggplot_sub,
       mapping = aes(x = sgRNA_type,y = celltype)) +
  geom_tile(aes(fill = roe_scale)) +
  geom_text(aes(label = label)) +
  guides(
    fill = guide_colorbar(title = 'Ro/e',title.vjust = 1)
  ) +
  scale_fill_gradient2(
      low = 'white',mid = colorRampPalette(colors = c('white','#5E8CBA'))(10)[8],
      high =  '#5E8CBA',midpoint = 1.6,limit = c(limit_min,limit_max)
  ) +
  theme_bw() +
  coord_flip() +
  theme(
    panel.grid = element_blank(),
    panel.border = element_blank(),
    axis.title = element_blank(),
    axis.line = element_blank(),
    axis.ticks = element_blank(),
    axis.text = element_text(size = 20),
    axis.text.x = element_text(angle = 30,hjust = 1),
    axis.text.y = element_text(hjust = 1),
    legend.text = element_text(size = 16),
    legend.position = 'right',
    legend.justification = c(0,1),
    legend.title = element_text(size = 20),
    strip.text = element_blank(),
    strip.background = element_blank()
  )
ggsave('./result_figs/Roe_Alod4_group.pdf',width = 6,height = 8)

### Fig3 j

In [ ]:
library(stringr)
library(Seurat)
library(Augur)
library(dplyr)
library(patchwork)
library(viridis)
library(qs)
library(BiocParallel)

In [ ]:
?Seurat::subset

In [ ]:
meta_data <- seurat_use@meta.data %>% 
    mutate(
        sgRNA_identity = sgRNA_identity %>% as.character(),
        sgRNA_identity = ifelse(sgRNA_identity %in% c('AAVS','NT1','NT2','NT'),yes = 'NT',no = sgRNA_identity),
        sgRNA_identity = factor(sgRNA_identity,levels = sgRNA_identity %>% unique())
    ) %>% 
    filter(group_alod4_30 != 'middle')
seurat_use_augur <- subset(seurat_use,cell = meta_data %>% rownames())
meta_data <- seurat_use_augur@meta.data %>% 
    mutate(
        sgRNA_identity = sgRNA_identity %>% as.character(),
        sgRNA_identity = ifelse(sgRNA_identity %in% c('AAVS','NT1','NT2','NT'),yes = 'NT',no = sgRNA_identity),
        sgRNA_identity = factor(sgRNA_identity,levels = sgRNA_identity %>% unique())
    ) %>% 
    filter(group_alod4_30 != 'middle')
seurat_use_augur@meta.data <- meta_data
meta_data$batch %>% table()
meta_data$sgRNA_identity %>% table()

In [ ]:
seurat_use_augur$batch %>% table()

In [ ]:
all(colnames(seurat_use_augur) %in% colnames(seurat_use_augur[["Protein"]]))
all(colnames(seurat_use_augur) %in% colnames(seurat_use_augur[["RNA"]]))

In [ ]:
is.na(seurat_use@meta.data$sgRNA_identity) %>% table()
seurat_use@meta.data$sgRNA_identity %>% unique()

In [ ]:
seurat_use_augur$group_alod4_30 %>% unique()

In [ ]:
DefaultAssay(seurat_use_augur) <- 'RNA'

In [ ]:
all(colnames(seurat_use_augur) %in% colnames(seurat_use_augur[["Protein"]]))
all(colnames(seurat_use_augur) %in% colnames(seurat_use_augur[["RNA"]]))

In [ ]:
library(tester)
library(pbmcapply)
library(purrr)
library(yardstick)
library(sparseMatrixStats)
library(tidyselect)
library(recipes)
library(rsample)
library(randomForest)
library(magrittr)

In [ ]:
calculate_auc_self <- function (
    input, meta = NULL, label_col = "label", cell_type_col = "cell_type", 
    n_subsamples = 50, subsample_size = 20, folds = 3, min_cells = NULL, 
    var_quantile = 0.5, feature_perc = 0.5, n_threads = 4, show_progress = T, 
    select_var = T, augur_mode = c("default", "velocity", "permute"), 
    classifier = c("rf", "lr"), 
    rf_params = list(trees = 100, mtry = 2, min_n = NULL, importance = "accuracy"), 
    lr_params = list(mixture = 1,penalty = "auto")
  ){
  classifier = match.arg(classifier)
  augur_mode = match.arg(augur_mode)
  if (n_subsamples > 1 & subsample_size/folds < 2) {
    stop("subsample_size / n_folds must be greater than or equal to 2")
  }
  if (is.null(min_cells)) {
    min_cells = subsample_size
  }
  if (classifier == "lr" && !requireNamespace("glmnet", quietly = TRUE)) {
    stop("install \"glmnet\" R package to run Augur with logistic regression ", 
         "classifier", call. = FALSE)
  }
  if ("Seurat" %in% class(input)) {
    if (!requireNamespace("Seurat", quietly = TRUE)) {
      stop("install \"Seurat\" R package for Augur compatibility with ", 
           "input Seurat object", call. = FALSE)
    }
    meta = input@meta.data %>% droplevels()
    cell_types = meta[[cell_type_col]]
    labels = meta[[label_col]]
    expr = Seurat::GetAssayData(input)
    default_assay = Seurat::DefaultAssay(input)
    message("using default assay: ", default_assay, " ...")
  }else if ("cell_data_set" %in% class(input)) {
    if (!requireNamespace("monocle3", quietly = TRUE)) {
      stop("install \"monocle3\" R package for Augur compatibility with ", 
           "input monocle3 object", call. = FALSE)
    }
    meta = monocle3::pData(input) %>% droplevels() %>% as.data.frame()
    cell_types = meta[[cell_type_col]]
    labels = meta[[label_col]]
    expr = monocle3::exprs(input)
  }else if ("SingleCellExperiment" %in% class(input)) {
    if (!requireNamespace("SingleCellExperiment", quietly = TRUE)) {
      stop("install \"SingleCellExperiment\" R package for Augur ", 
           "compatibility with input SingleCellExperiment object", 
           call. = FALSE)
    }
    meta = SummarizedExperiment::colData(input) %>% droplevels() %>% 
      as.data.frame()
    cell_types = meta[[cell_type_col]]
    labels = meta[[label_col]]
    expr = SummarizedExperiment::assay(input)
  }else {
    if (is.null(meta)) {
      stop("must provide metadata if not supplying a Seurat or monocle object")
    }
    valid_input = is(input, "sparseMatrix") || is_numeric_matrix(input) || is_numeric_dataframe(input)
    if (!valid_input) 
      stop("input must be Seurat, monocle, sparse matrix, numeric matrix, or ","numeric data frame")
    expr = input
    meta %<>% droplevels()
    cell_types = meta[[cell_type_col]]
    labels = meta[[label_col]]
  }
  if (!is.numeric(meta[[label_col]])) {
    meta[[label_col]] = as.factor(as.character(meta[[label_col]]))
    labels = meta[[label_col]]
  }
  if (length(dim(expr)) != 2 || !all(dim(expr) > 0)) {
    stop("expression matrix has at least one dimension of size zero")
  }
  n_cells1 = nrow(meta)
  n_cells2 = ncol(expr)
  if (n_cells1 != n_cells2) {
    stop("number of cells in metadata (", n_cells1, ") does not match number ", 
         "of cells in expression (", n_cells2, ")")
  }
  if (n_distinct(labels) == 1) {
    stop("only one label provided: ", unique(labels))
  }
  if (any(is.na(labels))) {
    stop("labels contain ", sum(is.na(labels)), "missing values")
  }
  if (any(is.na(cell_types))) {
    stop("cell types contain ", sum(is.na(cell_types)), "missing values")
  }
  if ("label" %in% rownames(expr)) {
    warning("row `label` exists in input; changing ...")
    to_fix = which(rownames(expr) == "label")
    rownames(expr)[to_fix] = paste0("label", seq_along(rownames(expr)[to_fix]))
  }
  rf_engine = "randomForest"
  if (classifier == "rf" && rf_engine == "ranger") {
    invalid_rows = any(grepl("-|\\.|\\(|\\)", rownames(expr)))
    if (invalid_rows) {
      warning("classifier `rf` with engine `ranger` cannot handle characters ", 
              "-.() in column names; replacing ...")
      expr %<>% set_rownames(gsub("-|\\.|\\(|\\)", "", 
                                  rownames(.)))
    }
  }
  missing = is.na(expr)
  if (any(missing)) {
    stop("matrix contains ", sum(missing), "missing values")
  }
  if (is.numeric(labels)) {
    mode = "regression"
    multiclass = F
    if (n_distinct(labels) <= 3) {
      warning("doing regression with only ", n_distinct(labels), 
              " unique values")
    }
  }else {
    mode = "classification"
    multiclass = n_distinct(labels) > 2
    if (multiclass & classifier == "lr") {
      stop("multi-class classification with classifier = 'lr' is currently not ", 
           "supported in tidymodels `logistic_reg`")
    }
    if (!is.factor(labels)) {
      warning("coercing labels to factor ...")
      labels %<>% as.factor()
    }
  }
  if (show_progress == T) {
    apply_fun = pbmclapply
  }else {
    apply_fun = mclapply
  }
  if (augur_mode == "velocity") {
    message("disabling feature selection for augur_mode=\"velocity\" ...")
    feature_perc = 1
    var_quantile = 1
  }else if (augur_mode == "permute" & n_subsamples < 100) {
    message("resetting n_subsamples from ", n_subsamples, 
            " to ", n_subsamples, " for augur_mode=\"permute\" ...")
    n_subsamples = 500
  }
  res = apply_fun(unique(cell_types), mc.cores = n_threads,function(cell_type) {
    # cell_type <- unique(cell_types)[1]
    y = labels[cell_types == cell_type]
    if (mode == "classification") {
      if (min(table(y)) < min_cells) {
        warning("skipping cell type ", cell_type, ": minimum number of cells (", min(table(y)), ") is less than ", min_cells)
        return(list())
      }
    }else if (mode == "regression") {
      if (length(y) < min_cells) {
        warning("skipping cell type ", cell_type, ": total number of cells (", length(y), ") is less than ", min_cells)
        return(list())
      }
    }
    X = expr[, cell_types == cell_type]
    min_features_for_selection = 1000
    if (nrow(X) >= min_features_for_selection & select_var) {
      X %<>% select_variance(var_quantile, filter_negative_residuals = F)
    }
    tmp_results = data.frame()
    tmp_importances = data.frame()
    seeded_rf <- function(form, data) {
      target_indexes = which(colnames(data) == form[[2]])
      y_var = data[, target_indexes] %>% t() %>% as.factor()
      x_var = data[, -target_indexes]
      original_seed <- .Random.seed
      set.seed(1)
      forest <- randomForest(
        y = y_var, x = x_var, 
        importance = TRUE, localImp = TRUE, 
        ntree = rf_params$trees, 
        mtry = rf_params$mtry, 
        min_n = rf_params$min_n, 
        type = mode)
      .Random.seed <- original_seed
      return(forest)
    }
    retrieve_class_preds = function(split, recipe, model) {
      test = bake(recipe, assessment(split))
      tbl = tibble(
        true = test$label, pred = predict(model,test),
        prob = predict(model, test, type = "prob")) %>%
        cbind(.$prob) %>% dplyr::select(-prob)
      colnames(tbl)[levels(test$label) == colnames(tbl)] %<>% paste0(".pred_", .)
      return(tbl)
    }
    
    retrieve_reg_preds = function(split, recipe, model) {
      test = bake(recipe, assessment(split))
      tbl = tibble(true = test$label, pred = predict(model,test)$.pred)
      return(tbl)
    }
    if (mode == "regression") {
      multi_metric = metric_set(
        ccc, huber_loss_pseudo,
        huber_loss, mae, mape, mase, rpd, rpiq, rsq_trad,
        rsq, smape, rmse)
    }else {
      multi_metric = metric_set(accuracy, precision,recall, sens, spec, npv, ppv, roc_auc)
    }
    prob_select = 3
    if (mode == "classification") {
      estimator = ifelse(multiclass, "macro", "binary")
      if (multiclass) 
        prob_select = seq(3, 3 + n_distinct(labels) -1)
      metric_fun = function(x) multi_metric(x, truth = true, estimate = pred, prob_select, estimator = estimator)
    }else {
      metric_fun = function(x) multi_metric(x, truth = true,estimate = pred, prob_select)
    }
    importance_name = "importance"
    if (mode == "regression") {
      if (rf_params$importance == "accuracy") {
        impval_name = "%IncMSE"
      }
      else {
        impval_name = "IncNodePurity"
      }
    }else {
      if (rf_params$importance == "accuracy") {
        impval_name = "MeanDecreaseAccuracy"
      }else {
        impval_name = "MeanDecreaseGini"
      }
    }
    if (rf_engine == "ranger") {
      importance_name = "variable.importance"
      impval_name == ".x[[i]]"
    }
    n_iter = ifelse(n_subsamples < 1, 1, n_subsamples)
    for (subsample_idx in seq_len(n_iter)) {
      set.seed(subsample_idx)
      if (augur_mode == "permute") {
        y = sample(y)
      }
      if (n_subsamples < 1) {
        if (nrow(X) >= min_features_for_selection & feature_perc < 1) {
          X0 = select_random(X, feature_perc)
        }else {X0 = X}
        X0 %<>% t() %>% as.matrix() %>% as.data.frame() %>%  repair_names() %>% mutate(label = y)
      }else {
        if (mode == "regression") {
          subsample_idxs = data.frame(label = y, position = seq_along(y)) %>% 
            do(sample_n(., subsample_size)) %>% pull(position)
        }else {
          subsample_idxs = data.frame(label = y, position = seq_along(y)) %>% 
            group_by(label) %>% do(sample_n(., subsample_size)) %>%
            pull(position)
        }
        y0 = y[subsample_idxs]
        if (nrow(X) >= min_features_for_selection & feature_perc < 1) {
          X0 = select_random(X, feature_perc)
        }else {
          X0 = X
        }
        X0 %<>% magrittr::extract(, subsample_idxs) %>% BiocGenerics::t() %>%
          magrittr::extract(, colVars(.) > 0) %>% as.matrix() %>% 
          as.data.frame() %>% repair_names() %>% mutate(label = y0)
      }
      if (classifier == "lr") {
        family = ifelse(multiclass, "multinomial", "binomial")
        if (is.null(lr_params$penalty) || lr_params$penalty == "auto") {
          lr_params$penalty = withCallingHandlers({
            glmnet::cv.glmnet(X0 %>% ungroup() %>%
                                select(-label) %>% as.matrix() %>% 
                                extract(,setdiff(colnames(.), "label")), X0$label,nfolds = folds, family = family) %>% 
              extract2("lambda.1se")
          }, warning = function(w) {
            if (grepl("dangerous ground", conditionMessage(w)))
              invokeRestart("muffleWarning")
          })
        }
        clf = logistic_reg(mixture = lr_params$mixture,penalty = lr_params$penalty, mode = "classification") %>% 
          set_engine("glmnet", family = family)
      }else if (classifier != "rf") {
        stop("invalid classifier: ", classifier)
      }
      if (mode == "classification") {
        cv = Augur:::vfold_cv(X0, v = folds, strata = "label")
      }else {
        cv = Augur:::vfold_cv(X0, v = folds)
      }
      withCallingHandlers({
        folded = cv %>% 
          mutate(
            recipes = splits %>% map(~prepper(., recipe = recipe(.$data, label ~ .))), 
            test_data = splits %>% map(analysis),
            fits = map2(recipes, test_data, ~seeded_rf(form = label ~ ., data = bake(object = .x, new_data = .y)))
        )
      }, warning = function(w) {
        if (grepl("dangerous ground", conditionMessage(w))) 
          invokeRestart("muffleWarning")
      })
      predictions = folded %>% mutate(pred = list(splits, recipes, fits))
      if (mode == "regression") {
        predictions = predictions %>% mutate(pred = pmap(pred,retrieve_reg_preds))
      }else {
        predictions = predictions %>% mutate(pred = pmap(pred,retrieve_class_preds))
      }
      eval = predictions %>% mutate(metrics = pred %>% map(metric_fun)) %>% extract2("metrics")
      result = eval %>% map2_df(., row.names(folded), ~mutate(.x, fold = .y))
      names(result) %<>% gsub("\\.", "", .)
      result %<>% mutate(cell_type = cell_type, subsample_idx = subsample_idx)
      importance = NULL
      if (classifier == "rf") {
        importance = folded %>% pull(fits) %>% map(importance_name) %>%
          map(as.data.frame) %>% 
          map( ~ rownames_to_column(.,"gene")) %>% 
          map2_df(1:length(.), ~mutate(.x,fold = .y)) %>% 
          mutate(cell_type = cell_type,subsample_idx = subsample_idx) %>% 
          dplyr::rename(importance = impval_name)
        importance %<>% dplyr::select(cell_type, subsample_idx,
                                      fold, gene, importance)
      }else if (classifier == "lr") {
        coefs = folded %>% pull(fits) %>% map("fit") %>%
          map( ~ as.matrix(coef(., s = lr_params$penalty)))
        sds = folded %>% pull(splits) %>% map("data") %>%
          map( ~ extract(., ,-ncol(.))) %>% map( ~ apply(.,2, sd))
        std_coefs = map(seq_len(folds), ~ coefs[[.]][-1,1] * sds[[.]])
        importance = std_coefs %>% map( ~ data.frame(gene = names(.),std_coef = .)) %>% 
          setNames(seq_len(folds)) %>%
          map2_df(names(.), ~ mutate(.x, fold = .y)) %>%
          mutate(cell_type = cell_type, subsample_idx = subsample_idx) %>%
          dplyr::select(cell_type, subsample_idx, fold,gene, std_coef)
      }
      result %<>% dplyr::select(cell_type, subsample_idx,fold, metric, estimator, estimate)
      tmp_results %<>% bind_rows(result)
      tmp_importances %<>% bind_rows(importance)
    }
    list(results = tmp_results, importances = tmp_importances)
  })
  if (any(map_lgl(res, ~"warning" %in% class(.)))) {
    res = res$value
  }
  if (all(lengths(res) == 0)) 
    stop("no cell type had at least ", min_cells, " cells in all conditions")
  print(res %>% head())
  if (mode == "classification") {
    AUCs = res %>% map("results") %>%
      bind_rows() %>% filter(metric == "roc_auc") %>% group_by(cell_type, subsample_idx) %>% 
      summarise(estimate = mean(estimate)) %>% ungroup() %>% 
      group_by(cell_type) %>% summarise(auc = mean(estimate)) %>% 
      ungroup() %>% arrange(desc(auc))
  }
  else if (mode == "regression") {
    CCCs = res %>% map("results") %>% bind_rows() %>% filter(metric == "ccc") %>% 
      group_by(cell_type, subsample_idx) %>% 
      summarise(estimate = mean(estimate)) %>% ungroup() %>% 
      group_by(cell_type) %>% summarise(ccc = mean(estimate)) %>% 
      ungroup() %>% arrange(desc(ccc))
  }
  feature_importances = res %>% map("importances") %>% bind_rows()
  results = res %>% map("results") %>% bind_rows()
  params = list(n_subsamples = n_subsamples, subsample_size = subsample_size, 
                folds = folds, min_cells = min_cells, var_quantile = var_quantile, 
                feature_perc = feature_perc, n_threads = n_threads, classifier = classifier)
  if (classifier == "rf") 
    params$rf_params = rf_params
  if (classifier == "lr") 
    params$lr_params = lr_params
  obj = list(X = expr, y = labels, cell_types = cell_types, 
             parameters = params, results = results, feature_importance = feature_importances)
  if (mode == "classification") {
    obj$AUC = AUCs
  }
  else if (mode == "regression") {
    obj$CCC = CCCs
  }
  return(obj)
}

In [ ]:
seurat_use_augur$group_alod4_30 %>% table()
seurat_use_augur$sgRNA_identity %>% table()
seurat_use_augur

In [ ]:
VariableFeatures(seurat_use_augur) %>% length()
data_input <- seurat_use_augur@assays$RNA$data %>% as.matrix() %>% as.data.frame() %>% 
    filter(rownames(.) %in% VariableFeatures(seurat_use_augur)) %>% 
    as.matrix()
data_input %>% head()

In [ ]:
augur <- calculate_auc_self(
    input = data_input,
    meta = seurat_use_augur@meta.data,
    var_quantile = 0.7,
    subsample_size = 10,show_progress = TRUE,
    cell_type_col = "sgRNA_identity", 
    label_col = "group_alod4_30", 
    n_threads = 16
)

In [ ]:
aucs <- augur$AUC
aucs$cell_type %>% unique()

In [ ]:
aucs <- augur$AUC %>% 
    mutate(
        auc = ifelse(auc>0.5,auc,1-auc),
        cell_type = cell_type %>% as.character()
    ) %>% 
    filter(
        cell_type %in% c(
            'NPC1-sg1','LDLR-sg1','INSIG1-sg1','NT','SOAT1-sg2','LDLR-sg3','ABCA1-sg1','NPC1-sg3','SREBF2-sg3',
            'SOAT1-sg3','NPC1-sg2','LDLR-sg2','SREBF2-sg1','HMGCR-sg3','HMGCR-sg1','SOAT1-sg1'
        )
    )
aucs %>% dim()
aucs %>% head()

In [ ]:
aucs

In [ ]:
options(repr.plot.width = 8,repr.plot.height = 9)
size_sm = 16
size_lg = 20
range = range(aucs$auc)
expand = abs(diff(range)) * 0.1
ggplot(data = aucs,aes(x = reorder(cell_type, auc), y = auc)) + 
    geom_segment(aes(xend = cell_type, yend = 0.49), size = 1) +
    geom_point(size = 28, aes(color = cell_type)) +
    geom_text(
        aes(label = format(auc,digits = 3), y = ifelse(auc < 0.5, 0.5, auc)), 
        size = 6, hjust = 0.5,color = 'white'#, nudge_y = expand
    ) +
    scale_y_continuous("AUC", limits = c(0.49, range[2] + expand*0.9),expand = c(0,0)) +#min(range[1] - expand, 0.5)
    scale_color_manual(values = c(
        '#59606d',
        '#555273',
        '#59606d','#59606d',
        '#65799b',
        '#59606d','#59606d',
        '#b6d5e1','#e2eff1'
    )) +
    coord_flip() + 
    theme_bw() + 
    theme(
        panel.border = element_blank(),
        panel.grid = element_blank(), 
        axis.text.x = element_text(size = size_sm), 
        axis.text.y = element_text(size = size_sm), 
        axis.title.x = element_text(size = size_lg), 
        axis.title.y = element_blank(), 
        strip.text = element_text(size = size_lg), strip.background = element_blank(), 
        # axis.line.y = element_blank(), axis.line.x = element_blank(), 
        axis.line = element_line(),
        legend.position = "none", 
        legend.text = element_text(size = size_sm), 
        legend.title = element_text(size = size_sm), 
        legend.key.size = unit(0.6,"lines"), 
        # legend.margin = margin(rep(0, 4)), 
        legend.background = element_blank(),
        plot.title = element_text(size = size_lg, hjust = 0.5)
    )
ggsave('./result_figs/AUC_Lollipop_sgRNAIdentity.pdf',width = 8,height = 9)

In [ ]:
table(seurat_use_augur$sgRNA_identity,seurat_use_augur$group_alod4_30)

In [ ]:
table(colnames(seurat_use_augur) == (seurat_use_augur@meta.data %>% rownames()))
all.equal(colnames(seurat_use_augur), rownames(seurat_use_augur@meta.data))

In [ ]:
getwd()

In [ ]:
saveRDS(seurat_use_augur,'./result_zxs/seurat_use_augur.rds')

### Fig2 c

In [ ]:
meta_data <- seurat_obj@meta.data %>% 
    mutate(
        batch = ifelse(batch == 'Tatin',yes = 'Tatin',no = 'Normal'),
        function_type = case_when(
            sgRNA_type %in% c("SREBF2", "HMGCR", "SQLE", "INSIG1") ~ 'chol_synthesis',
            sgRNA_type %in% c("LDLR", "NPC1L1", "NPC1") ~ 'chol_uptake',
            sgRNA_type %in% c("APOB", "MTTP", "ABCA1", "ABCG1", "ABCG5", "ABCG8", "NR1H3") ~ 'chol_efflux',
            sgRNA_type %in% c("SOAT1") ~ 'chol_esterification',
            sgRNA_type %in% c("AAVS",'NT') ~ 'NT',
            TRUE ~ sgRNA_type
        ),
        sgRNA_type_use = ifelse(sgRNA_type %in% c("AAVS",'NT'),yes = 'NT',no = sgRNA_type),
        sgRNA_type_use = factor(sgRNA_type_use,levels = c(
            "NT","SREBF2","HMGCR","SQLE","INSIG1","LDLR","NPC1L1","NPC1","APOB",
            "MTTP","ABCA1","ABCG1","ABCG5","ABCG8","NR1H3","SOAT1"
        ))
    )
seurat_obj@meta.data <- meta_data
meta_data$batch %>% table()
meta_data$sgRNA_type_use %>% unique()

In [ ]:
genes <- c("MYC", "PLK1", "CDK4", "CCNB1", "SOX2", "MDM2", 
           "MEF2C", "ATXN1", "ZEB2", "SNAI2", "SERPINE1", "POSTN", 
           "FADS2", "GPX4", "SLC7A11", "SQSTM1")
genes %>% length()

In [ ]:
seurat_obj$sgRNA_type %>% unique()

In [ ]:
tmp <- rownames(seurat_obj)
tmp[grepl('^SNA',tmp)]

In [ ]:
DefaultAssay(seurat_obj) <- 'RNA'
data_plot <- FetchData(seurat_obj, vars = c(genes, "batch",'sgRNA_type')) %>% 
    filter(sgRNA_type %in% c('AAVS','NT')) %>% 
    dplyr::select(-sgRNA_type) %>% arrange(batch) %>% 
    tidyr::pivot_longer(cols = all_of(genes),names_to = 'Genes',values_to = 'Expression') %>% 
    group_by(batch,Genes) %>% 
    summarise(
        Expression_mean = mean(Expression)
    ) %>% 
    pivot_wider(names_from = 'Genes',values_from = 'Expression_mean') %>% 
    column_to_rownames('batch') %>% t()
data_plot %>% dim()
data_plot %>% head()

In [ ]:
seurat_obj_subset <- subset(seurat_obj,subset = sgRNA_type %in% c('AAVS','NT'))
seurat_obj_subset

In [ ]:
seurat_obj_subset$batch %>% table()

In [ ]:
diffgene <- FindMarkers(
    object = seurat_obj_subset,
    slot = 'data',
    group.by = 'batch',
    ident.1	= 'Tatin',
    min.pct = 0,
    random.seed	= 1234,
    logfc.threshold = 0,
    only.pos = FALSE
) %>% 
    mutate(
        group = case_when(
            (p_val<0.05) & (avg_log2FC > 0.5) ~ 'Up in Tatin',
            (p_val<0.05) & (avg_log2FC < -0.5) ~ 'Down in Tatin',
            TRUE ~ 'Stable'
        )
    ) %>% 
    rownames_to_column('Genes')
diffgene$group %>% table()
diffgene %>% dim()
diffgene %>% head()

In [ ]:
data_plot <- diffgene %>% 
    filter(Genes %in% c(genes)) %>% 
    arrange(desc(avg_log2FC)) %>% 
    mutate(Genes = factor(Genes,levels = Genes %>% unique() %>% rev()))
data_plot$Genes %>% unique() %>% length()
data_plot

In [ ]:
data_plot$avg_log2FC %>% max()
data_plot$avg_log2FC %>% min()

In [ ]:
min_value <- data_plot$avg_log2FC %>% {. * 10 } %>% min() %>% ceiling() %>% {./10}
max_value <- data_plot$avg_log2FC %>% {. * 10 } %>% max() %>% floor() %>% {./10}
max_value

In [ ]:
options(repr.plot.width = 38,repr.plot.height = 1.5)
p <- ggplot(data = data_plot,aes(x = Genes,y = 1,fill = avg_log2FC)) +
    geom_tile(color = 'white') +
    geom_text(aes(label = Genes),size = 6,color = 'black') +
    scale_fill_gradient2(
        name = 'Log2(Fold Change)',
        low = '#418EA5',mid = '#FCFADD',high = '#CE584B',
        midpoint = 0.5,
        breaks = c(-1,-0.5,0,0.5,1,1.5),
        labels = c(-1,-0.5,0,0.5,1,1.5)
    ) +
    guides(
        fill = guide_colourbar(nrow = 1,title.position = 'top',hjust = 0.5)
    ) +
    labs(x = '', y= 'Tatin vs Normal') +
    theme_classic(base_size = 20) +
    theme(
        panel.border = element_rect(fill = NA),
        axis.line = element_blank(),
        # axis.text.x = element_text(angle = 90,hjust = 1,vjust = 0.5,size = 20),
        axis.text = element_blank(),
        axis.ticks.y = element_blank(),
        axis.title.y = element_text(angle = 0,hjust = 0.5,vjust = 0.5),
        legend.title = element_text(angle = 0,hjust = 0.5),
        legend.direction = 'horizontal',
        legend.key.height = unit(1.3, "cm"),
        legend.key.width = unit(1.5, "cm"),
        axis.title.x = element_blank()
    )
p
ggsave(plot = p,filename = './result_figs/Fig_4.3_mRNA_fc.pdf',width = 38,height = 1.5,limitsize = FALSE)

In [ ]:
library(ggplot2)
options(repr.plot.width = 24,repr.plot.height = 4)
p <- DotPlot(object = adata_sub_use,
        features = genes %>% unlist() %>% unique(),
        group.by = 'batch',scale = TRUE) +
  scale_color_gradient2(
    name = 'Average Expression',
    low = '#538CA9',mid = '#FCFADD',high = '#D97E73',
    breaks = c(-0.6,0,0.6),
    labels = c(-0.6,0,0.6)) +
  scale_size_continuous(name = 'Percent Expressed') +
  guides(
    color = guide_colourbar(nrow = 1,title.position = 'top'),
    size = guide_legend(ncol = 1,title.position = 'top')
  ) +
  labs(x = '', y= '') +
  theme(
    axis.text.x = element_text(angle = -45,hjust = 0,size = 20),
    axis.text.y = element_text(size = 20),
    legend.direction = 'vertical',
    axis.title.x = element_blank()
  )
p
ggsave(plot = p,filename = './result_figs/Fig_4.3_mRNA_dotplot.pdf',width = 24,height = 4,limitsize = FALSE)

In [ ]:
library(clusterProfiler)
library(msigdbr)
library(purrr)

In [ ]:
gene_list <- msigdbr(species = 'Homo sapiens')
gene_list_sub <- gene_list %>% 
    dplyr::select(c('gs_name','gene_symbol')) %>% 
    rename_all(~c('term','gene'))
gene_list_sub %>% head()

In [ ]:
adata_sub_use <- subset(seurat_obj,subset = sgRNA_type %in% c('AAVS','NT'))
adata_sub_use$sgRNA_identity %>% table()
adata_sub_use

In [ ]:
library(presto)
exp.genes <- wilcoxauc(adata_sub_use, 'batch')
dplyr::count(exp.genes, group) %>% print()
cluster0.genes<- exp.genes %>%
  filter(group == 'Tatin') %>%
  arrange(desc(logFC),desc(auc)) %>%
  dplyr::select(feature,logFC,auc)
ranks<- cluster0.genes$logFC
names(ranks) <- cluster0.genes$feature
geneList <- ranks
geneList %>% head()

In [ ]:
egmt_hallmark <- GSEA(
    geneList, 
    TERM2GENE=gene_list_sub,
    minGSSize = 5,
    pvalueCutoff = 0.8,
    verbose=FALSE
)

In [ ]:
gsea_result <- egmt_hallmark@result %>% 
    filter(p.adjust	<0.05) %>% 
    mutate(
        dbname = ID %>% str_split('_') %>% purrr::map_chr(~ .[1])
    ) %>% 
    filter(dbname %in% c('GOBP','HALLMARK'))
gsea_result %>% dim()
gsea_result %>% head()

In [ ]:
gsea_result %>% arrange(desc(NES))

In [ ]:
gsea_result %>% arrange(desc(NES)) %>% pull(Description)

In [ ]:
selected_pathways <- c(
  'GOBP_STEROL_METABOLIC_PROCESS','GOBP_STEROL_BIOSYNTHETIC_PROCESS','HALLMARK_CHOLESTEROL_HOMEOSTASIS',
    'GOBP_LIPID_HOMEOSTASIS','GOBP_LIPID_OXIDATION','GOBP_REGULATION_OF_AUTOPHAGY','GOBP_CHROMOSOME_SEGREGATION',
    'HALLMARK_MYC_TARGETS_V1','HALLMARK_E2F_TARGETS'
)


In [ ]:
tmp <- gsea_result$Description

In [ ]:
tmp[grepl('S6_KINASE',tmp)]

In [ ]:
data_plot_gsea <- gsea_result %>% 
    filter(Description %in% selected_pathways) %>% 
    arrange(desc(NES)) %>% 
    mutate(Description = factor(Description,levels = Description))
data_plot_gsea %>% dim()
data_plot_gsea %>% head()

In [ ]:
data_gene_diff <- geneList %>% 
    as.data.frame() %>% 
    rename_all(~c('lgfc')) %>% 
    rownames_to_column('gene')
data_gene_diff

In [ ]:
data_plot_gsea %>% head()

In [ ]:
# geneList
data_plot <- gene_list_sub %>% 
    filter(term %in% selected_pathways) %>% 
    left_join(data_gene_diff,by = 'gene') %>% 
    mutate(term = factor(term,levels = data_plot_gsea$Description)) %>% 
    left_join(
        data_plot_gsea %>% dplyr::rename(term = ID) %>% dplyr::select(term,NES,p.adjust),by = 'term'
    ) %>% 
    arrange(desc(NES)) %>% 
    mutate(term = factor(term,levels = term %>% unique() %>% rev()))
data_plot %>% head()

In [ ]:
data_plot %>% 
    pull(term)%>% unique()

In [ ]:
data_plot_clean <- data_plot[is.finite(data_plot$lgfc) & is.finite(data_plot$p.adjust), ]
data_plot_clean$lgfc %>% min()

In [ ]:
options(repr.plot.width = 14,repr.plot.height = 24)

library(ggh4x)
ggplot(data = data_plot_clean,aes(x = lgfc)) +
    geom_density(data = subset(data_plot_clean, lgfc > -0.3),aes(fill = -log10(p.adjust))) +
    geom_point(aes(x = -0.5,y = 0.5,color = NES, size = abs(NES))) +
    scale_fill_gradient2(
        name = '-Log10(p.adjust)',
        low = '#418EA5',mid = '#FCFADD',high = '#CE584B',
        midpoint = 2.2,breaks = c(1.6,2.2,2.8),labels = c(1.6,2.2,2.8)
    ) +
    scale_color_gradient2(
        name = 'NES',
        low = '#418EA5',mid = '#FCFADD',high = '#CE584B',
        midpoint = 0,breaks = c(-1,0,1),labels = c(-1,0,1)
    ) +
    scale_size_continuous(range = c(3,12)) +
    guides(fill = guide_legend(title.position = "left", title.theme = element_text(angle = 90))) +
    facet_wrap(~term,ncol = 1,scales = 'free_y',switch = 'left') +
    scale_x_continuous(limits = c(-0.5,0.4)) +
    theme_classic()+
    theme(
        plot.title = element_text(hjust = 0.5,size = 32),
        legend.title = element_text(size = 24,hjust = 0.5),
        legend.text = element_text(size = 20),
        legend.key.height = unit(1.9, "cm"),
        legend.key.width = unit(1.5, "cm"),
        axis.text.y = element_blank(),
        axis.ticks.y = element_blank(),
        axis.line.y = element_blank(),
        axis.text.x = element_text(size = 20,angle = 0,hjust = 0.5,vjust = 0.5),
        axis.title = element_blank(),
        strip.background = element_blank(),
        strip.text.y.left = element_text(size = 20,face = 'italic',vjust = 0.5,angle = 0,hjust = 1),
        strip.placement = 'outside'
    )

In [ ]:
data_plot %>% head()

In [ ]:
-log10(data_plot_clean$p.adjust) %>% max()
-log10(data_plot_clean$p.adjust) %>% min()
((-log10(data_plot_clean$p.adjust) %>% max()) + (-log10(data_plot_clean$p.adjust) %>% min()) )/2

In [ ]:
custom_palette <- colorRampPalette(c("#BEDBEF", "#2C5B92"))
custom_palette(3)

In [ ]:
library(ggridges) 
options(repr.plot.width = 14,repr.plot.height = 18)
ggplot(data_plot_clean, aes(x = lgfc,y = term)) +
	geom_density_ridges(aes(fill = -log10(p.adjust))) +
    geom_point(aes(x = -0.45,y = term,color = NES, size = abs(NES))) +
    geom_vline(xintercept = 0,linetype = 'longdash') +
    scale_fill_gradient2(
        name = '-Log10(p.adjust)',
        low = custom_palette(3)[1],mid = custom_palette(3)[2],high = custom_palette(3)[3],
        midpoint = 2.6,breaks = c(1.6,2.6,3.6),labels = c(1.6,2.6,3.6)
    ) +
    scale_color_gradient2(
        name = 'NES',
        low = '#418EA5',mid = '#FCFADD',high = '#CE584B',
        midpoint = 0,breaks = c(-1,0,1),labels = c(-1,0,1)
    ) +
    scale_size_continuous(range = c(3,12)) +
    scale_y_discrete(expand = c(0,0.1)) +
    guides(fill = guide_colorbar(title.position = "left", title.theme = element_text(angle = 90))) +
    labs(x = 'log2FoldChange') +
	theme(
        panel.grid = element_blank(),
        panel.background = element_blank(),
        panel.border = element_rect(color = "black", size = 1, fill = NA),
        plot.title = element_text(hjust = 0.5,size = 32),
        legend.title = element_text(size = 24,hjust = 0.5),
        legend.text = element_text(size = 20),
        legend.key.height = unit(1.9, "cm"),
        legend.key.width = unit(1.5, "cm"),
        axis.title.x = element_text(size = 24,hjust = 0.5,vjust = 0.5),
        axis.title.y = element_blank(),
        axis.text.y = element_text(size = 24,hjust = 1,vjust = 0.5),
        # axis.ticks.y = element_blank(),
        axis.line.y = element_blank(),
        axis.text.x = element_text(size = 20,angle = 0,hjust = 0.5,vjust = 0.5)
    )
ggsave(filename = './result_figs/Fig_ridges_gsea.pdf',width = 14,height = 18,limitsize = FALSE)

#### AUCell analysis of specific pathways for differentially expressed genes

In [ ]:
library(clusterProfiler)
library(msigdbr)
library(purrr)

In [ ]:
gene_list <- msigdbr(species = 'Homo sapiens')
gene_list_sub <- gene_list %>% 
    # filter(gs_name %in% pathway_select) %>% 
    select(c('gs_name','gene_symbol')) %>% 
    rename_all(~c('term','gene'))
gene_list_sub %>% head()

In [ ]:
adata_sub_use <- subset(seurat_obj,subset = sgRNA_type %in% c('AAVS','NT'))
adata_sub_use$sgRNA_identity %>% table()
adata_sub_use$batch %>% table()
adata_sub_use

In [ ]:
pathway_select <- c(
    "GOMF_RIBOSOMAL_PROTEIN_S6_KINASE_ACTIVITY","GOBP_PHOSPHATIDYLINOSITOL_3_KINASE_PROTEIN_KINASE_B_SIGNAL_TRANSDUCTION",
    "GOBP_CYTOKINE_MEDIATED_SIGNALING_PATHWAY","GOBP_CELL_MOTILITY","GOBP_CELL_CYCLE_DNA_REPLICATION","HALLMARK_MYC_TARGETS_V1",
    "GOBP_FERROPTOSIS","GOBP_REGULATION_OF_AUTOPHAGY","GOBP_CELLULAR_RESPONSE_TO_OXIDATIVE_STRESS",
    "HALLMARK_MYC_TARGETS_V2","GOBP_CELLULAR_RESPONSE_TO_REACTIVE_OXYGEN_SPECIES","GOBP_APOPTOTIC_PROCESS"
)
pathway_select %>% length()

In [ ]:
tmp <- gene_list_sub$term %>% unique()
tmp[(tmp %in% pathway_select)]
pathway_select[!(pathway_select %in% tmp)]

In [ ]:
# GOBP_PHOSPHATIDYLINOSITOL_3_KINASE_PROTE: GOBP_PHOSPHATIDYLINOSITOL_3_KINASE_PROTEIN_KINASE_B_SIGNAL_TRANSDUCTION
# tmp[grepl('PHOSPHATIDYLINOSITOL_3_KINASE_PROTE',tmp)]
# GO_CYTOKINE_METABOLIC_PROCESS: GOBP_CYTOKINE_MEDIATED_SIGNALING_PATHWAY
# tmp[grepl('GOBP_CYTOKINE',tmp)]
# GOBP_CELL_MIGRATION: GOBP_CELL_MOTILITY
tmp[grepl('GOBP_CELL_MOTILITY',tmp)]

In [ ]:
gene_list_use <- gene_list_sub %>% 
    filter(term %in% pathway_select) %>% 
    unstack(gene ~ term)
gene_list_use %>% names() %>% length()
gene_list_use %>% names()
gene_list_use[[1]]

In [ ]:
adata_sub_use <- AddModuleScore(
    adata_sub_use,
    features = gene_list_use,
    ctrl = 100,seed = 1234,
    name = "pathway_")

In [ ]:
meta_data_use <- adata_sub_use@meta.data %>% 
    rename_with(
        .fn = ~ names(gene_list_use),
        .cols = starts_with("pathway_")
    )
meta_data_use %>% head()
adata_sub_use@meta.data <- meta_data_use

In [ ]:
names(meta_data_use)

In [ ]:
data_plot <- meta_data_use %>% 
    select(-c(
        'orig.ident','nCount_RNA','nFeature_RNA','nCount_Protein','nFeature_Protein','nCount_sgRNA',
        'nFeature_sgRNA','sgRNA_identity','sgRNA_type','function_type','sgRNA_type_tmp','sgRNA_type_use'
    )) %>% 
    pivot_longer(cols = names(.)[-1],names_to = 'Pathway',values_to = 'AUcell_Score')
data_plot %>% head()

In [ ]:
library(ggpubr)
options(repr.plot.width = 54,repr.plot.height = 12)
ggplot(data_plot, aes(x = batch, y = AUcell_Score, fill = batch)) +
  geom_violin(position = position_dodge(width = 0.9), trim = FALSE) +
  geom_boxplot(width = 0.2, position = position_dodge(width = 0.9), outlier.shape = NA,show.legend = FALSE) +
  stat_compare_means(
    aes(group = color_use,fill = color_use),
    comparisons = list(
        c('Normal','Tatin')
    ),
    label = "p.signif",
    method = "wilcox.test",size = 8
  ) +
  scale_fill_manual(name = 'sgRNA Type',values = c('#96B6D8','#97C8AF'),) +
  theme_classic(base_size = 20) +
  facet_wrap(~Pathway,nrow = 1,strip.position = 'left',scales = 'free') +
  guides(fill = guide_legend(title.position = "left", title.theme = element_text(angle = 0))) +
  theme(
    plot.title = element_text(hjust = 0.5,size = 32),
    legend.title = element_text(size = 24),
    legend.text = element_text(size = 20),
    legend.key.height = unit(1.2, "cm"),
    legend.key.width = unit(1.2, "cm"),
    axis.text.x = element_blank(),
    axis.title = element_blank(),
    strip.background = element_blank(),
    strip.text = element_text(size = 20,face = 'italic',vjust = 0.7),
    strip.placement = 'outside'
  )


#### GSVA results of all pathways

In [ ]:
library(clusterProfiler)
library(msigdbr)
library(purrr)

In [ ]:
pathway_select <- c(
  "GOBP_CELL_CYCLE_G2_M_PHASE_TRANSITION",
  "HALLMARK_E2F_TARGETS",
  "HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION",
  "GOBP_FERROPTOSIS",
  "GOBP_REGULATION_OF_RESPONSE_TO_OXIDATIVE_STRESS",
  "HALLMARK_MYC_TARGETS_V2",
  "HALLMARK_TNFA_SIGNALING_VIA_NFKB",
  "HALLMARK_KRAS_SIGNALING_UP",
  "GOBP_NEGATIVE_REGULATION_OF_PROTEIN_MATURATION",
  "GOBP_POSITIVE_REGULATION_OF_TRANSFORMING_GROWTH_FACTOR_BETA1_PRODUCTION",
  "GOBP_RIBOSOME_BIOGENESIS"
)

In [ ]:
gene_list <- msigdbr(species = 'Homo sapiens')
gene_list_sub <- gene_list %>% 
    filter(gs_name %in% pathway_select) %>% 
    dplyr::select(c('gs_name','gene_symbol')) %>% 
    rename_all(~c('term','gene')) %>% 
    unstack(form = gene ~ term)
gene_list_sub %>% names() %>% head()

In [ ]:
# gene_list <- msigdbr(species = 'Homo sapiens')
# gene_list_sub <- gene_list %>% 
#     dplyr::select(c('gs_name','gene_symbol')) %>% 
#     rename_all(~c('term','gene')) %>% 
#     unstack(form = gene ~ term)
# gene_list_sub %>% names() %>% head()

In [ ]:
DefaultAssay(seurat_obj) <- 'RNA'
adata_sub_use <- subset(seurat_obj,subset = sgRNA_type %in% c('AAVS','NT'))
adata_sub_use$sgRNA_identity %>% table()
adata_sub_use$batch %>% table()
adata_sub_use

In [ ]:
expr <- adata_sub_use@assays$RNA@layers$data
expr <- FetchData(object = adata_sub_use,layer = 'data',vars = c(rownames(adata_sub_use))) %>% 
    t() %>% as.data.frame() %>% filter(rowSums(.)>0) %>% as.matrix()
expr %>% head()

In [ ]:
library(GSVA)
gsvapar <- gsvaParam(expr, gene_list_sub, maxDiff=TRUE)
gsva.res <- gsva(gsvapar) 

In [ ]:
data_plot <- gsva.res %>% as.data.frame() %>% 
    rownames_to_column('Pathway') %>% 
    pivot_longer(cols = names(.)[-1],names_to = 'sample',values_to = 'gsva_score') %>% 
    left_join(
        adata_sub_use@meta.data %>% dplyr::select('batch') %>% rownames_to_column('sample'),
        by = 'sample'
    )
data_plot %>% head()

In [ ]:
library(dplyr)
# 对每个 Pathway 计算 Wilcoxon P 值、中位数差、FDR 校正
pval_df <- data_plot %>%
  group_by(Pathway) %>%
  summarise(
    p_value = wilcox.test(gsva_score ~ batch)$p.value,
    median_diff = median(gsva_score[batch == "Tatin"]) - median(gsva_score[batch == "Normal"])
  ) %>%
  mutate(adj_p = p.adjust(p_value, method = "BH")) %>%
  arrange(p_value)  # 可选：按p值排序


In [ ]:
data_plot %>% 
    left_join(pval_df,by = c('Pathway')) %>% 
    group_by(batch,Pathway) %>% head()

In [ ]:
pval_df_filter <- pval_df %>% 
    # filter(adj_p<0.05) %>% 
    mutate(DBname = Pathway %>% str_split('_') %>% purrr::map_chr(~ .[1])) %>% 
    filter(DBname %in% c('GOMF','GOBP','GOCC','HALLMARK')) %>% 
    mutate(DBname = factor(DBname,levels = c('HALLMARK','GOBP','GOMF','GOCC'))) %>% 
    arrange(DBname,desc(median_diff))  
pval_df_filter %>% dim()
pval_df_filter %>% head()
# write.csv(pval_df,'./result_figs/data_gsva_pvalue.csv')

In [ ]:
pval_df %>% filter(grepl('Ferroptosis',Pathway,ignore.case = TRUE))

In [ ]:
pathway_select <- c(
  "GOBP_CELL_CYCLE_G2_M_PHASE_TRANSITION",
  "HALLMARK_E2F_TARGETS",
  "HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION",
  "GOBP_FERROPTOSIS",
  "GOBP_REGULATION_OF_RESPONSE_TO_OXIDATIVE_STRESS",
  "HALLMARK_MYC_TARGETS_V2",
  "HALLMARK_TNFA_SIGNALING_VIA_NFKB",
  "HALLMARK_KRAS_SIGNALING_UP",
  "GOBP_NEGATIVE_REGULATION_OF_PROTEIN_MATURATION",
  "GOBP_POSITIVE_REGULATION_OF_TRANSFORMING_GROWTH_FACTOR_BETA1_PRODUCTION",
  "GOBP_RIBOSOME_BIOGENESIS"
)
pathway_select %>% length()

In [ ]:
data_plot_use <- data_plot %>% 
    filter(Pathway %in% pathway_select)
data_plot_use$Pathway %>% unique() %>% length()

In [ ]:
library(ggpubr)
options(repr.plot.width = 54,repr.plot.height = 12)
ggplot(data_plot, aes(x = batch, y = gsva_score, fill = batch)) +
  geom_violin(position = position_dodge(width = 0.9), trim = FALSE) +
  geom_boxplot(width = 0.2, position = position_dodge(width = 0.9), outlier.shape = NA,show.legend = FALSE) +
  stat_compare_means(
    aes(group = batch,fill = batch),
    comparisons = list(
        c('Normal','Tatin')
    ),
    label = "p.signif",
    method = "wilcox.test",size = 8
    # position = position_dodge(width = 0.9)
  ) +
  scale_fill_manual(name = 'sgRNA Type',values = c('#96B6D8','#97C8AF'),) +
  theme_classic(base_size = 20) +
  facet_wrap(~Pathway,nrow = 1,strip.position = 'left',scales = 'free') +
  guides(fill = guide_legend(title.position = "left", title.theme = element_text(angle = 0))) +
  theme(
    plot.title = element_text(hjust = 0.5,size = 32),
    legend.title = element_text(size = 24),
    legend.text = element_text(size = 20),
    legend.key.height = unit(1.2, "cm"),
    legend.key.width = unit(1.2, "cm"),
    # axis.text.x = element_text(size = 20,angle = 90,hjust = 1,vjust = 0.5),
    axis.text.x = element_blank(),
    axis.title = element_blank(),
    strip.background = element_blank(),
    strip.text = element_text(size = 20,face = 'italic',vjust = 0.7),
    strip.placement = 'outside'
  )
# ggsave(plot = p,filename = './result_figs/Vlnplot_targetGene_3sgRNA-NT.pdf',width = 54,height = 14,limitsize = FALSE)

In [ ]:
data_plot %>% head()

In [ ]:
colours <- c("#FF0000","#00A08A")

ggplot(data_plot, aes(x = Pathway, y = gsva_score, fill = batch)) +
    introdataviz::geom_split_violin(alpha = .4) +    
    geom_boxplot(width = .15, alpha = .6, show.legend = FALSE) +    
    stat_summary(fun.data = "mean_se", geom = "pointrange", show.legend = F,
        position = position_dodge(.175)) +    
    scale_x_discrete(name = "Condition", labels = c("Non-word", "Word")) +    
    scale_y_continuous(name = "Reaction time (ms)") +    
    scale_fill_manual(values = colours, name = "Language group") +    
    theme_minimal()

In [ ]:
library(introdataviz)
options(repr.plot.width = 24,repr.plot.height = 12)
ggplot(data_plot, aes(x = Pathway, y = gsva_score, fill = batch)) + 
  # split violin
  geom_split_violin(alpha = 0.75,scale = 'width', trim = F,color = NA,width = 0.6) + 
  geom_boxplot(width = 0.15, alpha = .6, position = position_dodge(width = 0.2), outlier.shape = NA,show.legend = FALSE,color = 'white') +
  geom_text(aes(x = Pathway,y = 0,label = Pathway),angle = 90,hjust = 0.5,vjust = -3.5,size = 7) +
  stat_compare_means(
    aes(group = batch),
    # comparisons = list(
    #     c('Normal','Tatin')
    # ),
    label = "p.signif",
    method = "wilcox.test",size = 8
  ) +
  ###添加errorbar
  # stat_summary(data = data_plot, aes(x = Pathway,y = gsva_score, fill = batch),
  #              fun.min = function(x){quantile(x)[2]},
  #              fun.max = function(x){quantile(x)[4]},
  #              geom = 'errorbar', color = 'black',
  #              width = 0.1,size = 1,
  #              position = position_dodge(width = 0.2)) + 
  ylab("Value") + xlab(NULL) +
  scale_fill_manual(name = 'Group',values = c('#96B6D8','#97C8AF'),) +
  theme_classic(base_size = 20) +
  guides(fill = guide_legend(title.position = "left", title.theme = element_text(angle = 90))) +
  theme(
    plot.title = element_text(hjust = 0.5,size = 32),
    legend.title = element_text(size = 24,angle = 0),
    legend.text = element_text(size = 20),
    legend.key.height = unit(1.2, "cm"),
    legend.key.width = unit(1.2, "cm"),
    # axis.text.x = element_text(size = 20,angle = 90,hjust = 1,vjust = 0.5),
    axis.text.x = element_blank(),
    axis.title = element_blank()
  )
ggsave(filename = './result_figs/Fig_gsva_split_violin.pdf',width = 24,height = 12,limitsize = FALSE)

#### Expression of the protein

In [ ]:
genes <- c(
    "p-RPS6","pAKT","pP65","c-MYc","SOX2","Vimentin","P53","GPX4",'IntraIgG','surfaceIgG',"ALOD4"
) %>% paste(.,'-pAbO',sep = '')
genes

In [ ]:
seurat_obj %>% rownames()

In [ ]:
seurat_obj$sgRNA_type %>% unique()

In [ ]:
DefaultAssay(seurat_obj) <- 'Protein'
data_plot <- FetchData(seurat_obj, vars = c(genes, "batch",'sgRNA_type')) %>% 
    # mutate(color_use = sgRNA_identity %>% str_split('-') %>% purrr::map_chr(~ .[2])) %>% 
    filter(sgRNA_type %in% c('AAVS','NT')) %>% 
    dplyr::select(-sgRNA_type) %>% arrange(batch) %>% 
    tidyr::pivot_longer(cols = all_of(genes),names_to = 'Genes',values_to = 'Expression') %>% 
    group_by(batch,Genes) %>% 
    summarise(
        Expression_mean = mean(Expression)
    ) %>% 
    pivot_wider(names_from = 'Genes',values_from = 'Expression_mean') %>% 
    column_to_rownames('batch') %>% t()
data_plot %>% dim()
# data_plot_sgRNA$batch %>% unique()
data_plot %>% head()

In [ ]:
seurat_obj_subset <- subset(seurat_obj,subset = sgRNA_type %in% c('AAVS','NT'))
seurat_obj_subset

In [ ]:
diffgene <- FindMarkers(
    object = seurat_obj_subset,
    slot = 'data',
    group.by = 'batch',
    ident.1	= 'Tatin',
    min.pct = 0.00,
    random.seed	= 1234,
    logfc.threshold = 0,
    only.pos = FALSE
) %>% 
    mutate(
        group = case_when(
            (p_val<0.05) & (avg_log2FC > 0.5) ~ 'Up in Tatin',
            (p_val<0.05) & (avg_log2FC < -0.5) ~ 'Down in Tatin',
            TRUE ~ 'Stable'
        )
    ) %>% 
    rownames_to_column('Genes')
diffgene$group %>% table()
# diffgene %>% head()
diffgene[16:34,]

In [ ]:
data_plot <- diffgene %>% 
    filter(Genes %in% c(genes)) %>% 
    arrange(desc(avg_log2FC)) %>% 
    mutate(Genes = factor(Genes,levels = Genes %>% unique() %>% rev()))
data_plot

In [ ]:
data_plot$avg_log2FC %>% max()
data_plot$avg_log2FC %>% min()

In [ ]:
color_a <- '#9A8AB2'
color_b <- '#BA6B6E'
colorRampPalette(c(color_a,color_b))(10)

In [ ]:
options(repr.plot.width = 38,repr.plot.height = 1.5)
p <- ggplot(data = data_plot,aes(x = Genes,y = 1,fill = avg_log2FC)) +
    geom_tile(color = 'white') +
    geom_text(aes(label = Genes %>% str_remove('-pAbO')),size = 6,color = 'black') +
    scale_fill_gradient2(
        name = 'Log2(Fold Change)',
        # low = '#418EA5',mid = '#FCFADD',high = '#CE584B',
        low = colorRampPalette(c(color_a,color_b))(3)[1],mid = "white",high = colorRampPalette(c(color_a,color_b))(3)[3],
        midpoint = 0,breaks = c(-0.5,0,0.5,1),labels = c(-0.5,0,0.5,1)
    ) +
    guides(
        fill = guide_colourbar(nrow = 1,title.position = 'top',hjust = 0.5)
    ) +
    labs(x = '', y= 'Tatin vs Normal') +
    theme_classic(base_size = 20) +
    theme(
        panel.border = element_rect(fill = NA),
        axis.line = element_blank(),
        # axis.text.x = element_text(angle = 90,hjust = 1,vjust = 0.5,size = 20),
        axis.text = element_blank(),
        axis.ticks.y = element_blank(),
        axis.title.y = element_text(angle = 0,hjust = 0.5,vjust = 0.5),
        legend.title = element_text(angle = 0,hjust = 0.5),
        legend.direction = 'horizontal',
        legend.key.height = unit(1.3, "cm"),
        legend.key.width = unit(1.5, "cm"),
        axis.title.x = element_blank()
    )
p
ggsave(plot = p,filename = './result_figs/Fig_4.3_protein_fc.pdf',width = 38,height = 1.5,limitsize = FALSE)

In [ ]:
library(ggplot2)
options(repr.plot.width = 24,repr.plot.height = 4)
p <- DotPlot(object = seurat_obj_subset,
        features = genes %>% unlist() %>% unique(),
        group.by = 'batch',scale = TRUE) +
  scale_color_gradient2(
    name = 'Average Expression',
    # low = '#538CA9',mid = '#FCFADD',high = '#D97E73',
    low = colorRampPalette(c(color_a,color_b))(3)[1],mid = "white",high = colorRampPalette(c(color_a,color_b))(3)[3],
    breaks = c(-0.5,0,0.5,1),
    labels = c(-0.5,0,0.5,1)) +
  scale_size_continuous(name = 'Percent Expressed') +
  guides(
    color = guide_colourbar(nrow = 1,title.position = 'top'),
    size = guide_legend(ncol = 1,title.position = 'top')
  ) +
  labs(x = '', y= '') +
  theme(
    axis.text.x = element_text(angle = -45,hjust = 0,size = 20),
    axis.text.y = element_text(size = 20),
    legend.direction = 'vertical',
    axis.title.x = element_blank()
  )
p
ggsave(plot = p,filename = './result_figs/Fig_4.3_protein_dotplot.pdf',width = 24,height = 4,limitsize = FALSE)

#### Pathway-specific differential gene expression

In [ ]:
library(clusterProfiler)
library(msigdbr)
library(purrr)

In [ ]:
pathway_select <- c(
  "GOBP_CELL_CYCLE_G2_M_PHASE_TRANSITION",
  "HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION",
  "GOBP_FERROPTOSIS",
  "GOBP_REGULATION_OF_RESPONSE_TO_OXIDATIVE_STRESS",
  "HALLMARK_MYC_TARGETS_V2"
)

In [ ]:
gene_list <- msigdbr(species = 'Homo sapiens')
gene_list_sub <- gene_list %>% 
    filter(gs_name %in% pathway_select) %>% 
    dplyr::select(c('gs_name','gene_symbol')) %>% 
    rename_all(~c('term','gene'))
gene_list_sub %>% head()

In [ ]:
gene_list_sub$gene %>% unique() %>% length()

In [ ]:
genes <- gene_list_sub$gene %>% unique()
genes %>% length()

In [ ]:
seurat_obj$sgRNA_type %>% unique()

In [ ]:
DefaultAssay(seurat_obj) <- 'RNA'
all_vars <- colnames(FetchData(seurat_obj, vars = c(genes, "batch", "sgRNA_type")))
genes_present <- genes[genes %in% all_vars]
data_plot <- FetchData(seurat_obj, vars = c(genes_present, "batch",'sgRNA_type')) %>% 
    # mutate(color_use = sgRNA_identity %>% str_split('-') %>% purrr::map_chr(~ .[2])) %>% 
    filter(sgRNA_type %in% c('AAVS','NT')) %>% 
    dplyr::select(-sgRNA_type) %>% arrange(batch) %>% 
    tidyr::pivot_longer(cols = all_of(genes_present),names_to = 'Genes',values_to = 'Expression') %>% 
    group_by(batch,Genes) %>% 
    summarise(
        Expression_mean = mean(Expression)
    ) %>% 
    pivot_wider(names_from = 'Genes',values_from = 'Expression_mean') %>% 
    column_to_rownames('batch') %>% t()
data_plot %>% dim()
# data_plot_sgRNA$batch %>% unique()
data_plot %>% head()

In [ ]:
seurat_obj_subset <- subset(seurat_obj,subset = sgRNA_type %in% c('AAVS','NT'))
seurat_obj_subset

In [ ]:
diffgene <- FindMarkers(
    object = seurat_obj_subset,
    slot = 'data',
    group.by = 'batch',
    ident.1	= 'Tatin',
    min.pct = 0.00,
    random.seed	= 1234,
    logfc.threshold = 0,
    only.pos = FALSE
) %>% 
    mutate(
        group = case_when(
            (p_val<0.05) & (avg_log2FC > 0.5) ~ 'Up in Tatin',
            (p_val<0.05) & (avg_log2FC < -0.5) ~ 'Down in Tatin',
            TRUE ~ 'Stable'
        )
    ) %>% 
    rownames_to_column('Genes')
diffgene$group %>% table()
# diffgene %>% head()
diffgene[16:34,]

In [ ]:
data_plot <- diffgene %>% 
    filter(Genes %in% c(genes)) %>% 
    arrange(desc(avg_log2FC)) %>% 
    left_join(gene_list_sub %>% dplyr::rename('Genes' = 'gene'),by = 'Genes')
    # mutate(Genes = factor(Genes,levels = Genes %>% unique() %>% rev()))
data_plot

In [ ]:
data_plot$term %>% table()

In [ ]:
data_plot$avg_log2FC %>% max()
data_plot$avg_log2FC %>% min()

In [ ]:
options(repr.plot.width = 48,repr.plot.height = 1.5*20)
p <- ggplot(data = data_plot,aes(x = Genes,y = 1,fill = avg_log2FC)) +
    geom_tile(color = 'white') +
    # geom_text(aes(label = Genes %>% str_remove('-pAbO')),size = 6,color = 'black') +
    scale_fill_gradient2(
        name = 'Log2(Fold Change)',
        low = '#418EA5',mid = '#FCFADD',high = '#CE584B',
        midpoint = 0,breaks = c(-1.6,0,1.6),labels = c(-1.6,1,0.6)
    ) +
    guides(
        fill = guide_colourbar(nrow = 1,title.position = 'top',hjust = 0.5)
    ) +
    labs(x = '', y= 'Tatin vs Normal') +
    facet_wrap(~term,ncol = 1,scales = 'free',strip.position = 'top') +
    theme_classic(base_size = 20) +
    theme(
        panel.border = element_rect(fill = NA),
        axis.line = element_blank(),
        axis.text.x = element_text(angle = 90,hjust = 1,vjust = 0.5,size = 20),
        # axis.text = element_blank(),
        axis.ticks.y = element_blank(),
        axis.title.y = element_blank(),
        # axis.title.y = element_text(angle = 0,hjust = 0.5,vjust = 0.5),
        legend.title = element_text(angle = 0,hjust = 0.5),
        legend.direction = 'horizontal',
        legend.key.height = unit(1.3, "cm"),
        legend.key.width = unit(1.5, "cm"),
        axis.title.x = element_blank(),
        strip.background = element_blank(),
        strip.text = element_text(size = 20,face = 'italic',vjust = 0.7),
        strip.placement = 'outside'
    )
p
ggsave(plot = p,filename = './result_figs/Fig_4.3_all-mRNA_fc.pdf',width = 48,height = 30,limitsize = FALSE)

#### Heatmap of gene expression after filtering for specific pathways

In [ ]:
genes <- c(
  "MYC", "CDK4", "PLK1", "CCNB1",
  "CHEK1", "MELK", "BRCA1", "VIM",
  "ZEB2", "SNAI2", "CDH11", "SERPINE1",
  "POSTN", "FADS2", "KEAP1", "GPX4",
  "SQSTM1", "NQO1", "SLC7A11", "Sox2",
  "Runx2", "Kit", "MEF2C", "ATXN1", "MDM2"
) %>% str_to_upper()
genes %>% length()

In [ ]:
tmp <- rownames(seurat_obj)
genes[!(genes %in% tmp)]

In [ ]:
tmp[grepl('^ATX',tmp)]

In [ ]:
tmp[grepl('^SNA',tmp)]

In [ ]:
seurat_obj$sgRNA_type %>% unique()

In [ ]:
DefaultAssay(seurat_obj) <- 'RNA'
all_vars <- colnames(FetchData(seurat_obj, vars = c(genes, "batch", "sgRNA_type")))
genes_present <- genes[genes %in% all_vars]
data_plot <- FetchData(seurat_obj, vars = c(genes_present, "batch",'sgRNA_type')) %>% 
    # mutate(color_use = sgRNA_identity %>% str_split('-') %>% purrr::map_chr(~ .[2])) %>% 
    filter(sgRNA_type %in% c('AAVS','NT')) %>% 
    dplyr::select(-sgRNA_type) %>% arrange(batch) %>% 
    tidyr::pivot_longer(cols = all_of(genes_present),names_to = 'Genes',values_to = 'Expression') %>% 
    group_by(batch,Genes) %>% 
    summarise(
        Expression_mean = mean(Expression)
    ) %>% 
    pivot_wider(names_from = 'Genes',values_from = 'Expression_mean') %>% 
    column_to_rownames('batch') %>% t()
data_plot %>% dim()
# data_plot_sgRNA$batch %>% unique()
data_plot %>% head()

In [ ]:
seurat_obj_subset <- subset(seurat_obj,subset = sgRNA_type %in% c('AAVS','NT'))
seurat_obj_subset

In [ ]:
diffgene <- FindMarkers(
    object = seurat_obj_subset,
    slot = 'data',
    group.by = 'batch',
    ident.1	= 'Tatin',
    min.pct = 0.00,
    random.seed	= 1234,
    logfc.threshold = 0,
    only.pos = FALSE
) %>% 
    mutate(
        group = case_when(
            (p_val<0.05) & (avg_log2FC > 0.5) ~ 'Up in Tatin',
            (p_val<0.05) & (avg_log2FC < -0.5) ~ 'Down in Tatin',
            TRUE ~ 'Stable'
        )
    ) %>% 
    rownames_to_column('Genes')
diffgene$group %>% table()
# diffgene %>% head()
diffgene[16:34,]

In [ ]:
data_plot <- diffgene %>% 
    filter(Genes %in% c(genes)) %>% 
    arrange(desc(avg_log2FC))
    # mutate(Genes = factor(Genes,levels = Genes %>% unique() %>% rev()))
data_plot

In [ ]:
data_plot$avg_log2FC %>% max()
data_plot$avg_log2FC %>% min()

In [ ]:
options(repr.plot.width = 48,repr.plot.height = 1.5)
p <- ggplot(data = data_plot,aes(x = Genes,y = 1,fill = avg_log2FC)) +
    geom_tile(color = 'white') +
    geom_text(aes(label = Genes %>% str_remove('-pAbO')),size = 6,color = 'black') +
    scale_fill_gradient2(
        name = 'Log2(Fold Change)',
        low = '#418EA5',mid = '#FCFADD',high = '#CE584B',
        midpoint = 0,breaks = c(-0.8,0,0.8,1.5),labels = c(-0.8,0,0.8,1.5)
    ) +
    guides(
        fill = guide_colourbar(nrow = 1,title.position = 'top',hjust = 0.5)
    ) +
    labs(x = '', y= 'Tatin vs Normal') +
    # facet_wrap(~term,ncol = 1,scales = 'free',strip.position = 'top') +
    theme_classic(base_size = 20) +
    theme(
        panel.border = element_rect(fill = NA),
        axis.line = element_blank(),
        # axis.text.x = element_text(angle = 90,hjust = 1,vjust = 0.5,size = 20),
        axis.text = element_blank(),
        axis.ticks.y = element_blank(),
        # axis.title.y = element_blank(),
        axis.title.y = element_text(angle = 0,hjust = 0.5,vjust = 0.5),
        legend.title = element_text(angle = 0,hjust = 0.5),
        legend.direction = 'horizontal',
        legend.key.height = unit(1.3, "cm"),
        legend.key.width = unit(1.5, "cm"),
        axis.title.x = element_blank(),
        strip.background = element_blank(),
        strip.text = element_text(size = 20,face = 'italic',vjust = 0.7),
        strip.placement = 'outside'
    )
p
ggsave(plot = p,filename = './result_figs/Fig_4.3_select-mRNA_fc_final.pdf',width = 48,height = 1.5,limitsize = FALSE)

## Fig2 g

In [ ]:
DefaultAssay(seurat_obj) <- 'RNA'

In [ ]:
meta_data <- seurat_obj@meta.data
meta_data <- meta_data %>% 
    mutate(
        function_type = case_when(
            sgRNA_type %in% c("SREBF2", "HMGCR", "SQLE", "INSIG1") ~ 'chol_synthesis',
            sgRNA_type %in% c("LDLR", "NPC1L1", "NPC1") ~ 'chol_uptake',
            sgRNA_type %in% c("APOB", "MTTP", "ABCA1", "ABCG1", "ABCG5", "ABCG8", "NR1H3") ~ 'chol_efflux',
            sgRNA_type %in% c("SOAT1") ~ 'chol_esterification',
            sgRNA_type %in% c("AAVS",'NT') ~ 'NT',
            TRUE ~ sgRNA_type
        ),
        sgRNA_type_tmp = ifelse(sgRNA_type %in% c("AAVS",'NT'),yes = 'NT',no = sgRNA_type)
    )
seurat_obj@meta.data <- meta_data
meta_data$function_type %>% table()
meta_data$sgRNA_type_tmp %>% table()
meta_data %>% head()

In [ ]:
meta_data$batch %>% table()

### Tatin

In [ ]:
seurat_obj$batch %>% unique()
seurat_obj_subset <- subset(seurat_obj,subset = batch == 'Tatin')
meta_data <- seurat_obj_subset@meta.data
meta_data %>% head()

In [ ]:
seurat_obj_subset$sgRNA_identity %>% unique()

In [ ]:
diffgene_tatin <- lapply(seurat_obj_subset$sgRNA_identity %>% unique(),function(sgRNA_identity_select){
    print(paste('Calculating cluster',sgRNA_identity_select))
    seurat_obj_use <- subset(seurat_obj_subset,subset = sgRNA_identity %in% c(sgRNA_identity_select,'NT1','NT2','AAVS'))
    diff_gene <- FindMarkers(
        object = seurat_obj_use,
        slot = 'data',
        ident.1 = sgRNA_identity_select,
        group.by = 'sgRNA_identity',
        only.pos = TRUE
    ) %>% 
        filter(avg_log2FC>0.1,p_val<0.01) %>% 
        mutate(
            sgRNA_identity = sgRNA_identity_select,
            group = sgRNA_identity_select %>% str_remove('-sg\\d+'),
            cellNumber = seurat_obj_use@meta.data %>% filter(sgRNA_identity == sgRNA_identity_select) %>% nrow()
        )
    return(diff_gene)
}) %>% do.call(rbind,.) %>% 
    group_by(group,sgRNA_identity,cellNumber) %>% 
    summarise(Counts = n()) %>% 
    mutate(batch = 'Tatin') %>% 
    filter(!(group %in% c('NT1','NT2','AAVS')))

In [ ]:
diffgene_tatin %>% head()

### Normal

In [ ]:
seurat_obj$batch %>% unique()
seurat_obj_subset <- subset(seurat_obj,subset = batch == 'Normal')
meta_data <- seurat_obj_subset@meta.data
meta_data %>% head()

In [ ]:
seurat_obj_subset$sgRNA_identity %>% unique() %>% sort()

In [ ]:
diffgene_normal <- lapply(seurat_obj_subset$sgRNA_identity %>% unique(),function(sgRNA_identity_select){
    print(paste('Calculating cluster',sgRNA_identity_select))
    seurat_obj_use <- subset(seurat_obj_subset,subset = sgRNA_identity %in% c(sgRNA_identity_select,'NT1','NT2','AAVS'))
    diff_gene <- FindMarkers(
        object = seurat_obj_use,
        slot = 'data',
        ident.1 = sgRNA_identity_select,
        group.by = 'sgRNA_identity',
        only.pos = TRUE
    ) %>% 
        filter(avg_log2FC>0.1,p_val<0.01) %>% 
        mutate(
            sgRNA_identity = sgRNA_identity_select,
            group = sgRNA_identity_select %>% str_remove('-sg\\d+'),
            cellNumber = seurat_obj_use@meta.data %>% filter(sgRNA_identity == sgRNA_identity_select) %>% nrow()
        )
    return(diff_gene)
}) %>% do.call(rbind,.) %>% 
    group_by(group,sgRNA_identity,cellNumber) %>% 
    summarise(Counts = n()) %>% 
    mutate(batch = 'Normal') %>% 
    filter(!(group %in% c('NT1','NT2','AAVS')))

In [ ]:
data_plot <- rbind(diffgene_tatin,diffgene_normal) %>% 
    mutate(
        color_use = log2(Counts + 1),
        size_use = log2(cellNumber + 1)
    )
data_plot %>% head()

In [ ]:
getwd()
write.csv(data_plot,'./result_figs/data_diffgene_num.csv')

In [ ]:
data_plot$color_use %>% max()
data_plot$color_use %>% min()
data_plot$size_use %>% max()
data_plot$size_use %>% min()

In [ ]:
group_ranges <- data_plot %>%
  group_by(group) %>%
  summarise(
    x = (max(as.numeric(sgRNA_identity)) + min(as.numeric(sgRNA_identity)))/2,
    xmin = min(as.numeric(sgRNA_identity)) - 0.5,
    xmax = max(as.numeric(sgRNA_identity)) + 0.5
  ) %>%
  mutate(ymin = 0, ymax = 1)  

tile_plot <- ggplot(group_ranges) +
  geom_rect(
    aes(xmin = xmin, xmax = xmax, ymin = ymin, ymax = ymax, fill = group),
    alpha = 0.4, color = NA
  ) +
  geom_text(aes(label = group, x = x,y = 0.5),size = 8) +
  scale_fill_manual(values = setNames(scales::hue_pal()(length(unique(group_ranges$group))), unique(group_ranges$group))) +
  theme_void() +
  theme(legend.position = "none")
tile_plot

In [ ]:
mid_color <- grDevices::colorRampPalette(c("white", "black"))(8)[2]
print(mid_color)

In [ ]:
options(repr.plot.width = 48,repr.plot.height = 3)
data_plot <- data_plot %>%
  mutate(sgRNA_identity = factor(sgRNA_identity, levels = unique(sgRNA_identity)))

group_ranges <- data_plot %>%
  group_by(group) %>%
  summarise(start = min(as.numeric(sgRNA_identity)),
            end = max(as.numeric(sgRNA_identity)),
            mid = (start + end) / 2)

vline_positions <- group_ranges$end[-nrow(group_ranges)] + 0.5
point_plot <- ggplot(data_plot, aes(x = sgRNA_identity, y = batch)) +
  geom_point(
    aes(fill = color_use, size = size_use),
    shape = 21, color = "black", stroke = 0.5
  ) +
  geom_vline(
    xintercept = vline_positions, linetype = "dashed", color = "grey40"
  ) +
  geom_text(
    data = group_ranges,
    aes(x = mid, y = Inf, label = group),
    vjust = -0.5, size = 6, fontface = "italic", inherit.aes = FALSE
  ) +
  scale_size_continuous(
    name = 'Log2(cell Number+1+1)', range = c(2,12),
    breaks = c(5.5,6.5,7.5), labels = c(5.5,6.5,7.5)
  ) +
  scale_fill_gradient2(
    name = 'Log2(DEG Number+1)',
    low = 'white',high = 'black',
    mid = '#DADADA',
    midpoint = 7, 
    breaks = c(5.5,8), labels = c(5.5,8)
  ) +
  guides(
    fill = guide_colourbar(nrow = 1, title.position = 'top', hjust = 0.5),
    size = guide_legend(nrow = 1, title.position = 'top', hjust = 0.5)
  ) +
  labs(x = '', y= '') +
  theme_classic(base_size = 20) +
  theme(
    panel.border = element_rect(fill = NA),
    axis.line = element_blank(),
    axis.text.x = element_blank(),
    axis.ticks.x = element_blank(),
    axis.text.y = element_text(angle = 0, hjust = 1, vjust = 0.5, size = 20),
    axis.title.y = element_text(angle = 0, hjust = 0.5, vjust = 0.5),
    legend.title = element_text(angle = 0, hjust = 0.5),
    legend.direction = 'horizontal',
    legend.key.height = unit(1.3, "cm"),
    legend.key.width = unit(1.5, "cm")
  )

point_plot

In [ ]:
library(patchwork)
options(repr.plot.width = 48,repr.plot.height = 4)
combined_plot <- point_plot/tile_plot + 
  plot_layout(heights = c(1,0.3))

combined_plot <- combined_plot & theme(plot.margin = unit(c(0, 0, 0, 0), "pt"))
combined_plot <- combined_plot + plot_annotation() & 
  theme(plot.margin = unit(c(0, 0, 0, 0), "pt"))  
combined_plot
ggsave(filename = './result_figs/Fig_Dotplot_diffgeneNumber.pdf',width = 48,height = 4,limitsize = FALSE)

## suppl Fig10 d

### Tatin

In [ ]:
DefaultAssay(seurat_obj) <- 'RNA'

In [ ]:
seurat_obj$sgRNA_type_use %>% unique()

In [ ]:
sgRNA_type_select <- seurat_obj$sgRNA_type_use %>% unique() %>% as.character() %>% {.[!grepl('NT',.)]}#c("LDLR","NPC1","SREBF2","SOAT1")
sgRNA_type_select %>% length()
sgRNA_type_select

In [ ]:
seurat_obj$batch %>% unique()
seurat_obj_subset <- subset(seurat_obj,subset = batch == 'Tatin')
seurat_obj_subset <- subset(seurat_obj_subset,subset = sgRNA_type_use %in% c(sgRNA_type_select,'NT','AAVS'))
meta_data <- seurat_obj_subset@meta.data
meta_data %>% dim()
meta_data %>% head()

In [ ]:
seurat_obj_subset$batch %>% unique()
seurat_obj_subset$sgRNA_type_use %>% unique()

In [ ]:
diffgene_tatin <- lapply(sgRNA_type_select,function(sgRNA_type_select){
    print(paste('Calculating cluster',sgRNA_type_select))
    seurat_obj_use <- subset(seurat_obj_subset,subset = sgRNA_type_use %in% c(sgRNA_type_select,'NT'))
    diff_gene <- FindMarkers(
        object = seurat_obj_use,
        slot = 'data',
        ident.1 = sgRNA_type_select,
        group.by = 'sgRNA_type_use',
        only.pos = TRUE
    ) %>% 
        filter(avg_log2FC>0.1,p_val<0.01) %>% 
        mutate(
            sgRNA_type = sgRNA_type_select,
            cellNumber = seurat_obj_use@meta.data %>% filter(sgRNA_type_use == sgRNA_type_select) %>% nrow()
        )
    return(diff_gene)
}) 
names(diffgene_tatin) <- paste('Tatin_',sgRNA_type_select,sep = '')

In [ ]:
names(diffgene_tatin)
diffgene_tatin[[1]] %>% head()

### Normal

In [ ]:
seurat_obj$batch %>% unique()
seurat_obj_subset <- subset(seurat_obj,subset = batch == 'Normal')
seurat_obj_subset <- subset(seurat_obj_subset,subset = sgRNA_type_use %in% c(sgRNA_type_select,'NT'))
meta_data <- seurat_obj_subset@meta.data
meta_data %>% head()

In [ ]:
diffgene_normal <- lapply(sgRNA_type_select,function(sgRNA_type_select){
    print(paste('Calculating cluster',sgRNA_type_select))
    seurat_obj_use <- subset(seurat_obj_subset,subset = sgRNA_type_use %in% c(sgRNA_type_select,'NT'))
    diff_gene <- FindMarkers(
        object = seurat_obj_use,
        slot = 'data',
        ident.1 = sgRNA_type_select,
        group.by = 'sgRNA_type_use',
        only.pos = TRUE
    ) %>% 
        filter(avg_log2FC>0.1,p_val<0.01) %>% 
        mutate(
            sgRNA_type = sgRNA_type_select,
            cellNumber = seurat_obj_use@meta.data %>% filter(sgRNA_type_use == sgRNA_type_select) %>% nrow()
        )
    return(diff_gene)
}) 
names(diffgene_normal) <- paste('Normal_',sgRNA_type_select,sep = '')

In [ ]:
names(diffgene_tatin)
diffgene_tatin[[1]] %>% head()

In [ ]:
diffgene_all <- c(diffgene_tatin,diffgene_normal)
names(diffgene_all)

In [ ]:
go_res <- lapply(names(diffgene_all),function(group){
    print(paste('Calculating cluster',group))
    gene_symbol <- diffgene_all[[group]] %>% filter(avg_log2FC>0.1,p_val<0.01) %>% rownames_to_column('Genes') %>% pull(Genes)
    enrich.go <- enrichGO(
      gene = gene_symbol,
      OrgDb = 'org.Hs.eg.db',
      keyType = 'SYMBOL',
      ont = 'BP',
      pAdjustMethod = 'fdr',
      pvalueCutoff = 1,
      qvalueCutoff = 1,
      readable = FALSE
    )
    result <- enrich.go@result %>% 
        mutate(sgRNA_type = group)
    return(result)
}) %>% do.call(rbind,.)

In [ ]:
go_res %>% dim()
go_res %>% head()

In [ ]:
tmp <- go_res$Description %>% unique()
tmp[grepl('cholesterol metabolic process',tmp,ignore.case = TRUE)]

In [ ]:
data_use <- go_res %>%
  mutate(cluster_id = sgRNA_type)

binary_matrix <- data_use %>%
  filter(pvalue < 0.05) %>%
  dplyr::select(sgRNA_type, Description) %>%
  mutate(value = 1) %>%
  pivot_wider(names_from = Description, values_from = value, values_fill = 0) %>% 
  column_to_rownames("sgRNA_type")

jaccard_similarity <- function(x, y) {
  intersect_sum <- sum(x & y)
  union_sum <- sum(x | y)
  if (union_sum == 0) return(0)
  return(intersect_sum / union_sum)
}

cluster_ids <- rownames(binary_matrix)
sim_matrix <- matrix(0, nrow = length(cluster_ids), ncol = length(cluster_ids),
                     dimnames = list(cluster_ids, cluster_ids))

for (i in seq_along(cluster_ids)) {
  for (j in seq_along(cluster_ids)) {
    sim_matrix[i, j] <- jaccard_similarity(binary_matrix[i, ], binary_matrix[j, ])
  }
}

In [ ]:
sim_matrix %>% dim()
sim_matrix %>% head()

In [ ]:
library(ComplexHeatmap)
library(circlize)
library(dendextend)

In [ ]:
n <- nrow(sim_matrix)
non_diag_values <- sim_matrix[upper.tri(sim_matrix) | lower.tri(sim_matrix)]
max_non_diag <- max(non_diag_values, na.rm = TRUE)
diag(sim_matrix) <- max_non_diag
sim_matrix %>% max()

In [ ]:
col_fun <- colorRamp2(
  c(0, sim_matrix %>% max() %>% {./2}, sim_matrix %>% max() %>% {.*1.2}),
  c("white", "skyblue", "darkblue"))

In [ ]:
options(repr.plot.width = 12,repr.plot.height = 12)
ht <- Heatmap(
  sim_matrix,
  name = "Jaccard",
  col = col_fun,
  cluster_rows = TRUE,
  cluster_columns = TRUE,
  show_row_dend = TRUE,
  show_column_dend = TRUE,
  row_names_gp = gpar(fontsize = 6),
  column_names_gp = gpar(fontsize = 6)
)
ht_built <- draw(ht, merge_legend = TRUE)
row_dend <- row_dend(ht_built)
col_dend <- column_dend(ht_built)
row_hc <- as.hclust(row_dend)
col_hc <- as.hclust(col_dend)
k_row <- 4
k_col <- 4

row_dend <- as.dendrogram(row_hc)
col_dend <- as.dendrogram(col_hc)

row_dend_colored <- color_branches(row_dend, k = k_row)
col_dend_colored <- color_branches(col_dend, k = k_col)

row_groups <- cutree(row_hc, k = k_row)
col_groups <- cutree(col_hc, k = k_col)

In [ ]:
  cell_fun = function(j, i, x, y, width, height, fill) {
    grid.text(
      label = sprintf("%.2f", sim_matrix[i, j]),  
      x = x, y = y,
      gp = gpar(fontsize = 10, col = "black")
    )
  }

In [ ]:
col_fun <- colorRamp2(
  c(0, sim_matrix %>% max(na.rm = TRUE) %>% {./2}, sim_matrix %>% max(na.rm = TRUE) %>% {.*1.2}),
  colorRampPalette(c("white", "darkblue"))(10)[c(1,9,10)]
)

In [ ]:
diag(sim_matrix) <- NA

In [ ]:
options(repr.plot.width = 18,repr.plot.height = 18)
ht_all <- Heatmap(
  sim_matrix,
  name = "Jaccard",
  col = col_fun,na_col = "white",
  cell_fun = function(j,i,x,y,width,height,fill){
    if(i != j){
        grid.text(
            label = sprintf("%.2f",sim_matrix[i,j]),
            x=x,y = y,gp = gpar(fontsize = 10,col = "black")
        ) 
    } 
  },
  cluster_rows = row_dend_colored,
  cluster_columns = col_dend_colored,
  row_split = k_row,
  column_split = k_col,
  # rect_gp = gpar(col = "black", lwd = 0.5),  # 每格边框
  show_row_dend = TRUE,
  show_column_dend = TRUE,
  row_names_gp = gpar(fontsize = 26),
  column_names_gp = gpar(fontsize = 26),
  show_heatmap_legend = FALSE
)
pdf('./result_figs/Fig_4.4_jaccardIndex_sgRNA_mRNA.pdf',width = 18,height = 18)
draw(ht_all,padding = unit(c(20, 20, 20, 40), "mm"))
dev.off()
draw(ht_all,padding = unit(c(20, 20, 20, 40), "mm"))

## Fig2 j

In [ ]:
group_select <- c(
    'Normal_INSIG1','Normal_NPC1L1','Normal_NPC1','Normal_LDLR',
    'Tatin_SOAT1','Tatin_LDLR')
sim_matrix_sub <- sim_matrix[group_select,group_select]

In [ ]:
diag(sim_matrix_sub) <- max(sim_matrix_sub,na.rm = TRUE)

In [ ]:
colorRampPalette(c("#AEC6DB", "#789FC4"))(3)[1:3]

In [ ]:
col_fun <- colorRamp2(
  c(0, sim_matrix %>% max() %>% {./2}, sim_matrix %>% max() %>% {.*1.2}),
  c("#AEC6DB", "#93B2CF", "#789FC4"))

In [ ]:
col_fun <- colorRamp2(
  c(0, sim_matrix %>% max(na.rm = TRUE) %>% {./2}, sim_matrix %>% max(na.rm = TRUE) %>% {.*1.2}),
  colorRampPalette(c("white", "#789FC4"))(10)[c(1,9,10)]
)

In [ ]:
options(repr.plot.width = 10,repr.plot.height = 10)
ht_sub <- Heatmap(
    sim_matrix_sub,
    name = "Jaccard",
    col = col_fun,
    # na_col = "white",
    cell_fun = function(j,i,x,y,width,height,fill){
        if(i != j){
            grid.text(
                label = sprintf("%.2f",sim_matrix_sub[i,j]),
                x=x,y = y,gp = gpar(fontsize = 10,col = "black")
            )
        }  
    },
    width = unit(15, "cm"),
    height = unit(15, "cm"),
    rect_gp = gpar(col = "white", lwd = 0.5),  
    show_row_dend = TRUE,
    show_column_dend = TRUE,
    row_names_gp = gpar(fontsize = 22),
    column_names_gp = gpar(fontsize = 22),
    show_heatmap_legend = TRUE,
    heatmap_legend_param = list(
        title = 'Jaccard Similarity',
        title_gp = grid::gpar(fontsize = 24, fontface = "bold", hjust = -10, vjust = 0.5),
        title_position = 'leftcenter-rot',
        legend_direction = 'vertical',
        legend_height = unit(80, units = "mm"),
        labels_gp = grid::gpar(fontsize = 24, fontface = "bold", hjust = 0, vjust = 0.5)
    )
)
pdf('./result_figs/jaccardIndex_sgRNA_mRNA_select.pdf',width = 10,height = 10)
draw(ht_sub,padding = unit(c(20, 20, 20, 40), "mm"))
dev.off()
draw(ht_sub,padding = unit(c(20, 20, 20, 40), "mm"))

## endline